# SPARC-Seg on BRISC 2025 - Brain Tumour MRI

**Sparse Concept Reasoning for Segmentation** - reference implementation for the
LVR @ WACV 2027 submission (*Latent Visual Reasoning: Perception, Imagination,
and Multimodal Thought*).

| | |
|---|---|
| **Dataset** | BRISC 2025 - Brain Tumour MRI |
| **Modality** | MRI (magnetic) |
| **Size** | ~6000 slices across glioma / meningioma / pituitary / no-tumour |
| **Suggested Kaggle input** | `briscdataset/brisc2025` |
| **Expected runtime** | ~6 h for the full plan on a T4; ~1.2 h for the quick plan. |

### What this notebook runs

The model keeps a **working sketch** `S` - a persistent spatial state - and
revises it over `K` steps of block-coordinate proximal descent on an explicit
energy `E(z, S)`. Each revision is expressed as a **sparse combination of a
learned dictionary of visual concepts**, so at every step the state is a short,
named list of atoms rather than an undifferentiated activation.

The point of the paper is not that this segments well (it does). The point is
that we can **prove the state was causally load-bearing**, which is the bar the
LVR call explicitly sets:

> *"the intermediate representation should play a testable computational role,
> not merely coincide with an ordinary hidden activation."*

### Why this dataset

The most physically distinct modality of the three, which is what supports 'not just an RGB trick'. Tumour-free slices carry empty masks and are kept for the same reason as BUSI's normals.

**Loader note.** Attach the segmentation task folder. The loader pairs images/ with masks/ under train/ and test/, recovering the subtype from the filename for stratification.

---

### Setup on Kaggle

1. **Add Input** -> search the dataset above -> Add.
2. **Settings -> Accelerator -> GPU T4 x2** (one is used; two is fine).
3. **Settings -> Internet -> ON** if you want ImageNet weights downloaded
   automatically. With internet OFF, attach any torchvision-weights dataset and
   the backbone loader finds it; failing both, it falls back to random
   initialisation and says so loudly.
4. Run all. Paths are discovered by file-signature matching, so a differently
   named mirror of the dataset still resolves.


In [ ]:
# Environment check. Nothing here installs anything: the Kaggle
# python image already ships every dependency this notebook uses.
import sys, platform, warnings
warnings.filterwarnings('ignore')
import numpy, torch, cv2, scipy, sklearn, matplotlib
print('python      ', platform.python_version())
print('torch       ', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu         ', torch.cuda.get_device_name(0))
print('numpy/cv2   ', numpy.__version__, cv2.__version__)

import os
if os.path.isdir('/kaggle/input'):
    print('\nmounted Kaggle datasets:')
    for p in sorted(os.listdir('/kaggle/input')):
        print('  -', p)
else:
    print('\n/kaggle/input not present - running outside Kaggle. '
          'Pass root=... to the driver below.')

---
## Part 1 - Shared reasoning core

Everything below this line is **identical in all three dataset notebooks**. It is
generated from one source package by `tools/build_notebooks.py`, so the three
dataset owners cannot drift apart - which is the condition the cross-modality
comparison depends on.

If you need to change the core, change it in the package and rebuild all three
notebooks. Do not hand-edit these cells.

Core integrity hash: `dc20f5afa23ccc07`


---
### The method, in the order the code implements it

**Energy.**

$$E(z,S)=\underbrace{\tfrac12\lVert S-Dz\rVert_F^2}_{\text{concept consistency}}
+\underbrace{\iota_{\mathcal C}(z)}_{\text{sparsity}}
+\underbrace{\lambda_2 R(S)}_{\text{shape prior}}
+\underbrace{\lambda_3\lVert S-g_\phi(x)\rVert^2}_{\text{image evidence}}$$

**Updates** (alternating; $\eta=1/L$, $L=\lVert D^\top D\rVert_2$ by power iteration):

$$z_{t+1}=P_{\mathcal C}\big(z_t-\eta D^\top(Dz_t-S_t)\big),\qquad
S_{t+1}=S_t-\eta' \nabla_S\big[\text{smooth part of } E\big]$$

**Three corrections to the original proposal**, each of which a reviewer would
otherwise have found first:

1. **The code is spatial.** The proposal wrote $S\in\mathbb R^{H\times W\times d}$
   but $z\in\mathbb R^m$; those are dimensionally incompatible inside
   $\lVert S-Dz\rVert_F$. Here $z\in\mathbb R^{B\times m\times h\times w}$ and $D$
   is a $1\times1$ convolution.

2. **The shape prior is smooth.** A persistent-homology penalty is
   piecewise-linear, so its gradient has no finite Lipschitz constant and the
   descent lemma's hypothesis simply fails - the stated Proposition would be
   false as written. Inside $E$ we use a Huber-smoothed total-variation plus
   curvature term, whose gradient *is* Lipschitz; Betti numbers are reported as
   an **evaluation** metric, where non-differentiability costs nothing. At
   evaluation the $S$-step additionally uses **Armijo backtracking**, which
   guarantees monotone descent for any finite local $L$ without having to assert
   a hand-derived value. `monotone_descent_rate` audits this numerically on
   every run - it should read exactly 1.000.

3. **Sparsity is a projection, not a penalty.** A fixed $\lambda_1$ controls
   sparsity only relative to the scale of the data it acts on, and both $S$ and
   $D$ change scale during training. Measured: a $\lambda_1$ giving ~4 active
   atoms at initialisation gave **47 of 48** after training - the audit story
   evaporated silently. $\mathcal C=\{z: \text{at most } k \text{ atoms active}\}$
   is scale-free, needs no per-dataset tuning, and keeps the guarantee intact:
   for any *closed* set and $\eta\le1/L$, projected gradient descends
   (Blumensath & Davies' IHT argument, at group granularity).

   This also makes the central control exact: in top-$k$ mode the sparsity term
   is the indicator of a set, contributing $0$ at every iterate. **SPARC-Seg and
   the dense control therefore minimise a numerically identical energy with
   identical parameters and identical $K$** - they differ only in the feasible
   set the $z$-step projects onto.


### 1.1 Configuration

Every number a reviewer might ask about lives in one place. `CoreConfig` is shared verbatim across the three datasets; only `in_channels` and the loader differ.

In [ ]:
# ===== sparcseg/config.py ====================================================
"""Central configuration for SPARC-Seg.

Every number a reviewer might ask about lives here, not scattered through the
code.  ``SHARED`` holds the values that MUST be identical across the three
dataset forks -- the cross-modality claim in the paper is only apples-to-apples
if these never diverge.  ``DATASETS`` holds the parts that are legitimately
dataset-specific (input channels, class semantics, where the files live).
"""

from __future__ import annotations

import dataclasses
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple


# --------------------------------------------------------------------------
# Shared reasoning-core hyperparameters.
#
# Per the "fixed (m, K)" decision: these are constant across BUSI / ISIC /
# BRISC for the main results table.  A per-dataset-tuned variant is available
# by overriding at call time, and is reported only as a supplementary ablation.
# --------------------------------------------------------------------------
@dataclass
class CoreConfig:
    # -- geometry -----------------------------------------------------------
    img_size: int = 256
    sketch_stride: int = 4          # reasoning runs at img_size / stride
    sketch_dim: int = 64            # d, channels of the working sketch S
    dict_size: int = 192            # m, number of concept atoms (m >> d)

    # -- reasoning loop -----------------------------------------------------
    n_steps: int = 4                # K

    # Sparsity control. "topk" projects onto {at most topk_atoms active}, which
    # is scale-free and therefore stable across datasets and across training;
    # "l1" is the classic penalty, kept as an ablation. A fixed lambda_1 was
    # measured to drift from ~4 active atoms at init to ~47/48 after training,
    # which is why the penalty is NOT the default.
    sparsity_mode: str = "topk"     # "topk" | "l1"
    topk_atoms: int = 8             # the "handful of atoms" the audit story needs
    straight_through: bool = True   # training-only; lets dead atoms be revived
    lambda_l1: float = 0.05         # used only when sparsity_mode == "l1"
    lambda_group: float = 0.02      # used only when sparsity_mode == "l1"

    lambda_topo: float = 0.10       # smooth shape-prior weight
    # The evidence pull is a BOTTLENECK PARAMETER, not a nuisance: as it grows,
    # S -> g(x) and the code stops mattering, i.e. the reasoning state becomes
    # decorative by construction. 0.25 keeps the code load-bearing while still
    # anchoring S to the image. ``code_explained_variance`` audits this.
    lambda_evidence: float = 0.25
    nonneg_code: bool = True        # z >= 0 -> atoms read as "concept present"

    # Step sizes.  The z-step uses the provably safe eta = 1/L with
    # L = ||D^T D||_2 estimated by power iteration.  The S-step uses a learned
    # base step during training and Armijo backtracking at evaluation time.
    s_step_init: float = 0.50
    backtracking_eval: bool = True
    backtrack_shrink: float = 0.5
    backtrack_max: int = 8
    armijo_c: float = 1e-4

    # -- adaptive depth -----------------------------------------------------
    adaptive_depth: bool = True
    energy_plateau_eps: float = 1e-3   # relative |dE| threshold for early exit
    min_steps: int = 1
    max_steps: int = 6                 # probed by the efficiency sweep only
    # Training samples the unroll depth uniformly from {1..n_steps}. Without it
    # the network is only ever optimised at exactly K steps, and running it at
    # any other depth degrades -- which would make the adaptive-depth claim an
    # artefact of the training schedule rather than a property of the method.
    train_depth_sampling: bool = True

    # -- readout / supervision ---------------------------------------------
    deep_supervision: bool = True
    deep_supervision_decay: float = 0.5  # weight_t = decay ** (K - t)

    # -- backbone -----------------------------------------------------------
    backbone: str = "resnet34"
    pretrained: bool = True
    freeze_bn: bool = False

    # -- optimisation -------------------------------------------------------
    epochs: int = 60
    batch_size: int = 8
    lr: float = 3e-4
    lr_backbone_mult: float = 0.1
    weight_decay: float = 1e-4
    amp: bool = True
    grad_clip: float = 1.0
    warmup_epochs: int = 2
    dice_ce_alpha: float = 0.5      # loss = a * BCE + (1 - a) * soft-Dice

    # -- protocol -----------------------------------------------------------
    n_folds: int = 5
    seeds: Tuple[int, ...] = (0, 1, 2)
    val_frac_within_train: float = 0.15
    num_workers: int = 2

    # -- causal-faithfulness protocol --------------------------------------
    # Energy fractions at which norm-matched ablation is evaluated.  These
    # define the necessity/sufficiency curves; CSI is the area between the
    # importance-ordered curve and the random-direction null.
    ablation_fractions: Tuple[float, ...] = (0.05, 0.1, 0.2, 0.3, 0.5, 0.7)
    n_random_controls: int = 8      # random directions per image per fraction
    steering_alphas: Tuple[float, ...] = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0)
    n_transplant_pairs: int = 200
    causal_max_images: int = 300    # cap for the intervention sweep (runtime)

    # -- evaluation ---------------------------------------------------------
    boundary_tolerances: Tuple[int, ...] = (2, 5)
    prob_threshold: float = 0.5
    bootstrap_n: int = 5000

    def replace(self, **kw: Any) -> "CoreConfig":
        return dataclasses.replace(self, **kw)


@dataclass
class DatasetConfig:
    key: str
    name: str
    modality: str
    physics: str
    in_channels: int = 3
    # Substrings used to locate the dataset under /kaggle/input without
    # hard-coding a Kaggle slug (slugs differ between mirrors of the same set).
    discovery_hints: Sequence[str] = field(default_factory=tuple)
    # Regex applied to a mask path to recover the matching image path.
    notes: str = ""


DATASETS: Dict[str, DatasetConfig] = {
    "busi": DatasetConfig(
        key="busi",
        name="BUSI (Breast Ultrasound Images)",
        modality="Ultrasound",
        physics="acoustic",
        in_channels=3,
        discovery_hints=("busi", "dataset_busi", "breast-ultrasound", "breast_ultrasound"),
        notes=(
            "780 images in benign/malignant/normal. 'normal' carries an all-zero "
            "mask. A minority of cases ship multiple mask files (_mask_1, _mask_2) "
            "which must be unioned, not silently dropped."
        ),
    ),
    "isic": DatasetConfig(
        key="isic",
        name="ISIC 2018 Task 1 (Skin Lesion Segmentation)",
        modality="Dermoscopy",
        physics="optical",
        in_channels=3,
        discovery_hints=("isic2018", "isic-2018", "isic_2018", "isic"),
        notes=(
            "Task 1 provides 2594 train / 100 val / 1000 test images with "
            "_segmentation.png masks. Official splits are used when present."
        ),
    ),
    "brisc": DatasetConfig(
        key="brisc",
        name="BRISC 2025 (Brain Tumor MRI Segmentation)",
        modality="MRI",
        physics="magnetic",
        in_channels=3,
        discovery_hints=("brisc", "brisc2025", "brisc-2025", "brain-tumor"),
        notes=(
            "Segmentation split holds images/ and masks/ under train/ and test/. "
            "Slices without tumour carry empty masks and are kept: they are the "
            "cases where a decorative reasoning state fails loudest."
        ),
    ),
}


# --------------------------------------------------------------------------
# Method registry -- what appears as a row in the main results table.
# --------------------------------------------------------------------------
METHODS: List[str] = [
    "singleshot",     # B1: same backbone, one forward pass, no reasoning loop
    "dense_unrolled", # B2: same loop, lambda_l1 = lambda_group = 0 (the control)
    "ptea_lite",      # B3: energy-based test-time refinement (PTEA re-impl)
    "sparcseg",       # ours
]

METHOD_LABELS: Dict[str, str] = {
    "singleshot": "Single-shot U-Net (no reasoning)",
    "dense_unrolled": "Dense unrolled refinement (lambda_1 = 0)",
    "ptea_lite": "Energy test-time adaptation (PTEA-lite)",
    "sparcseg": "SPARC-Seg (ours)",
    "vlm_cot": "Textual chain-of-thought VLM -> mask",
}

SHARED = CoreConfig()

In [ ]:
# ===== sparcseg/utils.py =====================================================
"""Small shared helpers: determinism, device handling, timers, JSON-safe dumps."""

from __future__ import annotations

import json
import os
import random
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seed every RNG we touch.

    ``deterministic`` trades a little throughput for run-to-run reproducibility,
    which matters here because the causal-faithfulness numbers are differences
    of differences -- non-determinism shows up directly in the reported effect.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_params(module: torch.nn.Module, trainable_only: bool = True) -> int:
    ps = module.parameters()
    if trainable_only:
        ps = (p for p in ps if p.requires_grad)
    return sum(p.numel() for p in ps)


@contextmanager
def timer(label: str, sink: Optional[Dict[str, float]] = None, verbose: bool = False):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        if sink is not None:
            sink[label] = sink.get(label, 0.0) + dt
        if verbose:
            print(f"[timer] {label}: {dt:.2f}s")


def to_jsonable(obj: Any) -> Any:
    """Recursively convert numpy / torch scalars so ``json.dump`` stops complaining."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    if isinstance(obj, Path):
        return str(obj)
    if hasattr(obj, "__dataclass_fields__"):
        return {k: to_jsonable(getattr(obj, k)) for k in obj.__dataclass_fields__}
    return obj


def save_json(obj: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)
    return path


def load_json(path: str | Path) -> Any:
    with open(path) as f:
        return json.load(f)


def banner(text: str, width: int = 78, char: str = "=") -> None:
    print("\n" + char * width)
    print(text)
    print(char * width)


def human_time(seconds: float) -> str:
    if seconds < 90:
        return f"{seconds:.1f}s"
    if seconds < 5400:
        return f"{seconds / 60:.1f}min"
    return f"{seconds / 3600:.2f}h"


class AverageMeter:
    """Running mean that ignores NaNs (empty-mask Dice can legitimately be NaN)."""

    def __init__(self) -> None:
        self.total = 0.0
        self.count = 0

    def update(self, value: float, n: int = 1) -> None:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return
        self.total += float(value) * n
        self.count += n

    @property
    def avg(self) -> float:
        return self.total / self.count if self.count else float("nan")


def infer_flops_proxy(n_steps: float, per_step_cost: float, base_cost: float) -> float:
    """Compute proxy used in the efficiency table: backbone + K reasoning steps."""
    return base_cost + n_steps * per_step_cost

In [ ]:
# ===== sparcseg/metrics.py ===================================================
"""Segmentation and topology metrics, computed per image so that paired
statistical tests are possible downstream.

Empty-mask convention (this matters: BUSI 'normal' and BRISC tumour-free slices
are legitimately empty, and silently dropping them inflates Dice):

    GT empty, pred empty      -> Dice = IoU = 1.0
    GT empty, pred non-empty  -> Dice = IoU = 0.0
    GT non-empty              -> usual formula

``aggregate`` reports both the all-image mean and the lesion-present-only mean,
because the two answer different questions and reviewers will ask for both.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:  # SciPy is present on Kaggle images; guard anyway so import never kills a run.
    from scipy import ndimage as ndi
    _HAVE_SCIPY = True
except Exception:  # pragma: no cover
    _HAVE_SCIPY = False


EPS = 1e-7


# --------------------------------------------------------------------------
# Overlap metrics
# --------------------------------------------------------------------------
def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if not gt.any():
        return 1.0 if not pred.any() else 0.0
    inter = np.logical_and(pred, gt).sum()
    return float(2.0 * inter / (pred.sum() + gt.sum() + EPS))


def iou_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if not gt.any():
        return 1.0 if not pred.any() else 0.0
    union = np.logical_or(pred, gt).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / (union + EPS))


def precision_recall(pred: np.ndarray, gt: np.ndarray) -> Tuple[float, float]:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    prec = tp / (pred.sum() + EPS) if pred.any() else (1.0 if not gt.any() else 0.0)
    rec = tp / (gt.sum() + EPS) if gt.any() else (1.0 if not pred.any() else 0.0)
    return float(prec), float(rec)


# --------------------------------------------------------------------------
# Boundary metrics -- the ones the paper's thesis actually rests on
# --------------------------------------------------------------------------
def _boundary_mask(mask: np.ndarray) -> np.ndarray:
    """1-pixel-wide inner boundary of a binary mask."""
    mask = mask.astype(bool)
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    if _HAVE_SCIPY:
        eroded = ndi.binary_erosion(mask, border_value=0)
        return np.logical_and(mask, ~eroded)
    # crude fallback: 4-neighbour difference
    b = np.zeros_like(mask)
    b[:-1, :] |= mask[:-1, :] != mask[1:, :]
    b[:, :-1] |= mask[:, :-1] != mask[:, 1:]
    return np.logical_and(b, mask)


def _dist_to(mask: np.ndarray) -> np.ndarray:
    """Euclidean distance from every pixel to the nearest True pixel of ``mask``."""
    if not mask.any():
        return np.full(mask.shape, np.inf, dtype=np.float32)
    if _HAVE_SCIPY:
        return ndi.distance_transform_edt(~mask).astype(np.float32)
    raise RuntimeError("boundary metrics require SciPy")


def boundary_f_score(pred: np.ndarray, gt: np.ndarray, tolerance: int = 2) -> float:
    """BF-score (Csurka et al., 2013): F1 of boundary pixels matched within
    ``tolerance`` pixels. This is the metric that separates 'roughly right blob'
    from 'right boundary', which is precisely the workshop's framing."""
    pb = _boundary_mask(pred)
    gb = _boundary_mask(gt)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    d_gb = _dist_to(gb)
    d_pb = _dist_to(pb)
    prec = float((d_gb[pb] <= tolerance).mean())
    rec = float((d_pb[gb] <= tolerance).mean())
    if prec + rec == 0:
        return 0.0
    return float(2 * prec * rec / (prec + rec))


def hd95(pred: np.ndarray, gt: np.ndarray, spacing: float = 1.0) -> float:
    """95th-percentile symmetric Hausdorff distance. NaN when either side is
    empty (undefined, not zero -- averaging zeros there would be a lie)."""
    pb = _boundary_mask(pred)
    gb = _boundary_mask(gt)
    if not pb.any() or not gb.any():
        return float("nan")
    d_gb = _dist_to(gb)
    d_pb = _dist_to(pb)
    fwd = d_gb[pb]
    bwd = d_pb[gb]
    return float(np.percentile(np.concatenate([fwd, bwd]), 95) * spacing)


# --------------------------------------------------------------------------
# Topology
# --------------------------------------------------------------------------
def betti0(mask: np.ndarray, connectivity: int = 2) -> int:
    """Number of connected components (beta_0)."""
    mask = mask.astype(bool)
    if not mask.any():
        return 0
    if not _HAVE_SCIPY:
        raise RuntimeError("betti0 requires SciPy")
    structure = ndi.generate_binary_structure(2, connectivity)
    _, n = ndi.label(mask, structure=structure)
    return int(n)


def betti1(mask: np.ndarray) -> int:
    """Number of holes (beta_1) via Euler characteristic: b1 = b0 - chi."""
    mask = mask.astype(bool)
    if not mask.any():
        return 0
    b0 = betti0(mask)
    holes = ndi.binary_fill_holes(mask)
    n_hole_px = np.logical_and(holes, ~mask)
    if not n_hole_px.any():
        return 0
    structure = ndi.generate_binary_structure(2, 1)
    _, n = ndi.label(n_hole_px, structure=structure)
    return int(n)


def betti_error(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    return {
        "betti0_err": float(abs(betti0(pred) - betti0(gt))),
        "betti1_err": float(abs(betti1(pred) - betti1(gt))),
    }


# --------------------------------------------------------------------------
# Per-image bundle + aggregation
# --------------------------------------------------------------------------
def all_metrics(
    pred: np.ndarray,
    gt: np.ndarray,
    boundary_tolerances: Sequence[int] = (2, 5),
    with_topology: bool = True,
) -> Dict[str, float]:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    prec, rec = precision_recall(pred, gt)
    out: Dict[str, float] = {
        "dice": dice_score(pred, gt),
        "iou": iou_score(pred, gt),
        "precision": prec,
        "recall": rec,
        "hd95": hd95(pred, gt),
        "gt_empty": float(not gt.any()),
        "pred_area": float(pred.sum()),
        "gt_area": float(gt.sum()),
    }
    for t in boundary_tolerances:
        out[f"bf{t}"] = boundary_f_score(pred, gt, tolerance=t)
    if with_topology and _HAVE_SCIPY:
        out.update(betti_error(pred, gt))
    return out


def aggregate(per_image: List[Dict[str, float]]) -> Dict[str, float]:
    """Mean over images, plus lesion-present-only means for the overlap metrics."""
    if not per_image:
        return {}
    keys = sorted({k for d in per_image for k in d})
    out: Dict[str, float] = {}
    for k in keys:
        vals = np.array([d.get(k, np.nan) for d in per_image], dtype=np.float64)
        finite = vals[np.isfinite(vals)]
        out[k] = float(finite.mean()) if finite.size else float("nan")
        out[f"{k}_std"] = float(finite.std(ddof=1)) if finite.size > 1 else 0.0
    present = [d for d in per_image if not d.get("gt_empty", 0.0)]
    if present and len(present) != len(per_image):
        for k in ("dice", "iou", "bf2", "bf5", "hd95"):
            vals = np.array([d[k] for d in present if k in d], dtype=np.float64)
            finite = vals[np.isfinite(vals)]
            if finite.size:
                out[f"{k}_present"] = float(finite.mean())
    out["n_images"] = float(len(per_image))
    return out


def column(per_image: List[Dict[str, float]], key: str) -> np.ndarray:
    """Extract one metric as an aligned vector -- the input to paired tests."""
    return np.array([d.get(key, np.nan) for d in per_image], dtype=np.float64)

In [ ]:
# ===== sparcseg/stats.py =====================================================
"""Paired statistics for the results tables.

The claims in this paper are all *relative* ("sparse states show a larger
causal gap than dense ones"), which makes paired tests on per-image scores the
right instrument -- not unpaired comparisons of two means.  Everything here
operates on aligned per-image vectors produced by ``metrics.column``.
"""

from __future__ import annotations

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

try:
    from scipy import stats as sps
    _HAVE_SCIPY = True
except Exception:  # pragma: no cover
    _HAVE_SCIPY = False


def _clean_pair(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError(f"paired vectors must align: {a.shape} vs {b.shape}")
    ok = np.isfinite(a) & np.isfinite(b)
    return a[ok], b[ok]


def bootstrap_ci(
    values: np.ndarray,
    n_boot: int = 5000,
    alpha: float = 0.05,
    seed: int = 0,
    statistic=np.mean,
) -> Tuple[float, float, float]:
    """Percentile bootstrap CI of a statistic. Returns (point, lo, hi)."""
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, v.size, size=(n_boot, v.size))
    boots = statistic(v[idx], axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(statistic(v)), float(lo), float(hi)


def paired_bootstrap_diff(
    a: np.ndarray, b: np.ndarray, n_boot: int = 5000, alpha: float = 0.05, seed: int = 0
) -> Dict[str, float]:
    """CI of the paired mean difference (a - b). Resamples *image indices*, so
    the pairing is preserved -- this is what makes the interval honest."""
    a, b = _clean_pair(a, b)
    if a.size == 0:
        return {"diff": float("nan"), "lo": float("nan"), "hi": float("nan"), "n": 0}
    d = a - b
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, d.size, size=(n_boot, d.size))
    boots = d[idx].mean(axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"diff": float(d.mean()), "lo": float(lo), "hi": float(hi), "n": int(d.size)}


def wilcoxon(a: np.ndarray, b: np.ndarray) -> Dict[str, float]:
    """Wilcoxon signed-rank test plus a rank-biserial effect size.

    Dice distributions are bounded, skewed and often have ties at 0 or 1, so a
    paired t-test is the wrong default; the signed-rank test is standard in the
    medical-segmentation literature for exactly this reason.
    """
    a, b = _clean_pair(a, b)
    if a.size < 3:
        return {"stat": float("nan"), "p": float("nan"), "effect": float("nan"), "n": int(a.size)}
    d = a - b
    nz = d[d != 0]
    if nz.size == 0:
        return {"stat": 0.0, "p": 1.0, "effect": 0.0, "n": int(a.size)}
    if _HAVE_SCIPY:
        stat, p = sps.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    else:  # normal approximation fallback
        ranks = np.argsort(np.argsort(np.abs(nz))) + 1.0
        w_plus = ranks[nz > 0].sum()
        n = nz.size
        mu = n * (n + 1) / 4.0
        sigma = np.sqrt(n * (n + 1) * (2 * n + 1) / 24.0)
        z = (w_plus - mu) / (sigma + 1e-12)
        stat, p = float(w_plus), float(2 * (1 - 0.5 * (1 + np.math.erf(abs(z) / np.sqrt(2)))))
    # rank-biserial correlation: a scale-free effect size in [-1, 1]
    ranks = np.argsort(np.argsort(np.abs(nz))) + 1.0
    total = ranks.sum()
    effect = float((ranks[nz > 0].sum() - ranks[nz < 0].sum()) / (total + 1e-12))
    return {"stat": float(stat), "p": float(p), "effect": effect, "n": int(a.size)}


def holm_bonferroni(pvalues: Sequence[float], alpha: float = 0.05) -> Dict[str, List]:
    """Holm step-down correction. Returns adjusted p-values and reject flags.

    With 4 methods x 3 datasets x several metrics the family-wise error rate is
    not negligible; reporting raw p-values across that grid invites a reviewer
    to discount the whole table.
    """
    p = np.asarray(pvalues, dtype=np.float64)
    m = p.size
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=np.float64)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)
        adjusted[idx] = min(1.0, running)
    return {
        "p_adjusted": adjusted.tolist(),
        "reject": (adjusted < alpha).tolist(),
    }


def compare_methods(
    per_method: Dict[str, np.ndarray],
    reference: str,
    n_boot: int = 5000,
    alpha: float = 0.05,
    seed: int = 0,
) -> Dict[str, Dict[str, float]]:
    """Compare every method against ``reference`` with paired tests + Holm."""
    if reference not in per_method:
        raise KeyError(f"reference method {reference!r} not in {list(per_method)}")
    others = [k for k in per_method if k != reference]
    rows: Dict[str, Dict[str, float]] = {}
    pvals: List[float] = []
    for k in others:
        w = wilcoxon(per_method[reference], per_method[k])
        ci = paired_bootstrap_diff(per_method[reference], per_method[k], n_boot, alpha, seed)
        rows[k] = {**ci, "p_raw": w["p"], "effect": w["effect"]}
        pvals.append(w["p"])
    if pvals:
        corr = holm_bonferroni(pvals, alpha=alpha)
        for k, padj, rej in zip(others, corr["p_adjusted"], corr["reject"]):
            rows[k]["p_holm"] = float(padj)
            rows[k]["significant"] = bool(rej)
    return rows


def stars(p: float) -> str:
    if not np.isfinite(p):
        return ""
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "n.s."


def spearman(x: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    """Spearman rank correlation (used for concept naming and the
    energy-vs-error diagnostic)."""
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if x.size < 3 or np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return {"rho": float("nan"), "p": float("nan"), "n": int(x.size)}
    if _HAVE_SCIPY:
        rho, p = sps.spearmanr(x, y)
        return {"rho": float(rho), "p": float(p), "n": int(x.size)}
    rx = np.argsort(np.argsort(x)).astype(np.float64)
    ry = np.argsort(np.argsort(y)).astype(np.float64)
    rho = float(np.corrcoef(rx, ry)[0, 1])
    return {"rho": rho, "p": float("nan"), "n": int(x.size)}

In [ ]:
# ===== sparcseg/data/discovery.py ============================================
"""Locate datasets under /kaggle/input without hard-coding a Kaggle slug.

Kaggle mirrors the same dataset under many different slugs and nests the actual
files at unpredictable depths.  Hard-coded paths are the single most common
reason a shared notebook fails on a teammate's machine, so everything here works
by *signature matching*: we look for the file-naming pattern the dataset is
known by, then take its parent directory as the root.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence

SEARCH_ROOTS: List[str] = [
    "/kaggle/input",
    "/kaggle/working/data",
    "./data",
    "../input",
    ".",
]

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def candidate_roots(extra: Optional[Sequence[str]] = None) -> List[Path]:
    roots: List[Path] = []
    for r in list(extra or []) + SEARCH_ROOTS:
        p = Path(r)
        if p.is_dir():
            roots.append(p)
    return roots


def list_input_datasets() -> List[Path]:
    """Top-level attached Kaggle datasets, for the 'what did I actually mount?'
    printout at the top of every notebook."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return []
    return sorted(p for p in base.iterdir() if p.is_dir())


def walk_files(root: Path, max_files: int = 400_000) -> Iterable[Path]:
    n = 0
    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        for fn in filenames:
            if Path(fn).suffix.lower() in IMAGE_EXTS:
                yield Path(dirpath) / fn
                n += 1
                if n >= max_files:
                    return


def find_by_signature(
    predicate: Callable[[Path], bool],
    hints: Sequence[str] = (),
    extra_roots: Optional[Sequence[str]] = None,
    min_hits: int = 5,
) -> Optional[Path]:
    """Return the deepest directory that contains >= ``min_hits`` files matching
    ``predicate``.  ``hints`` only reorders the search (preferring plausibly
    named mounts first); it never restricts it, so an oddly named upload still
    resolves.
    """
    roots = candidate_roots(extra_roots)
    scan_order: List[Path] = []
    for root in roots:
        subs = [p for p in root.iterdir() if p.is_dir()] if root.is_dir() else []
        preferred = [p for p in subs if any(h in p.name.lower() for h in hints)]
        rest = [p for p in subs if p not in preferred]
        scan_order.extend(preferred + rest + [root])

    counts: Dict[Path, int] = {}
    for start in scan_order:
        for f in walk_files(start):
            if predicate(f):
                counts[f.parent] = counts.get(f.parent, 0) + 1
        if counts and sum(counts.values()) >= min_hits:
            break

    if not counts:
        return None
    # Take the shallowest common ancestor of all hit directories: for BUSI this
    # lands on the folder holding benign/ malignant/ normal/ rather than on one
    # class folder.
    hit_dirs = [d for d, c in counts.items() if c > 0]
    common = Path(os.path.commonpath([str(d) for d in hit_dirs]))
    return common


def describe_mount(root: Optional[Path]) -> str:
    if root is None:
        return "NOT FOUND"
    n = sum(1 for _ in walk_files(root, max_files=50_000))
    return f"{root}  ({n} image files)"


class DatasetNotFound(RuntimeError):
    """Raised with an actionable message listing what *is* mounted."""

    def __init__(self, name: str, hints: Sequence[str]):
        mounted = list_input_datasets()
        listing = "\n".join(f"    - {p.name}" for p in mounted) or "    (nothing mounted)"
        super().__init__(
            f"Could not locate {name} under /kaggle/input.\n"
            f"  Looked for directories/files matching: {list(hints)}\n"
            f"  Currently mounted Kaggle datasets:\n{listing}\n"
            f"  Fix: click 'Add Input' in the Kaggle sidebar and attach the dataset, "
            f"or pass root=... explicitly to the loader."
        )

In [ ]:
# ===== sparcseg/data/common.py ===============================================
"""Dataset base class, augmentation and splitting.

Augmentation is implemented directly on top of OpenCV rather than through
albumentations: the Kaggle image ships several albumentations versions whose
APIs differ, and a silent augmentation difference between three teammates'
notebooks would quietly break the cross-modality comparison.
"""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


@dataclass
class Sample:
    """One image with its (possibly multiple) mask files."""
    image_path: Path
    mask_paths: List[Path]
    group: str = ""          # class / subtype label, used for stratification
    split_hint: str = ""     # 'train' / 'val' / 'test' when the dataset ships splits
    sample_id: str = ""

    def __post_init__(self) -> None:
        if not self.sample_id:
            self.sample_id = self.image_path.stem


# --------------------------------------------------------------------------
# I/O
# --------------------------------------------------------------------------
def read_image(path: Path, in_channels: int = 3) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:  # cv2 silently returns None on unreadable files
        raise FileNotFoundError(f"could not read image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if in_channels == 1:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[..., None]
    return img


def read_mask_union(paths: Sequence[Path], shape: Tuple[int, int]) -> np.ndarray:
    """Union of every mask file for a case.

    BUSI ships ``*_mask_1.png`` / ``_mask_2.png`` for multi-lesion cases.
    Taking only the first file (the common shortcut) silently deletes lesions
    and depresses recall for every method equally -- which hides real
    differences rather than revealing them.
    """
    out = np.zeros(shape, dtype=np.uint8)
    for p in paths:
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if m.shape != shape:
            m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
        out = np.maximum(out, (m > 127).astype(np.uint8))
    return out


# --------------------------------------------------------------------------
# Augmentation
# --------------------------------------------------------------------------
@dataclass
class AugConfig:
    hflip: float = 0.5
    vflip: float = 0.2
    rot90: float = 0.0            # off by default: anatomy has a canonical up
    scale_limit: float = 0.15
    shift_limit: float = 0.08
    rotate_limit: float = 20.0
    affine_p: float = 0.7
    brightness_limit: float = 0.2
    contrast_limit: float = 0.2
    photometric_p: float = 0.5
    gamma_limit: Tuple[float, float] = (0.8, 1.25)
    gamma_p: float = 0.3
    elastic_p: float = 0.0        # available but off: it fights topology metrics


def _affine(img: np.ndarray, mask: np.ndarray, cfg: AugConfig, rng: np.random.Generator):
    h, w = img.shape[:2]
    angle = rng.uniform(-cfg.rotate_limit, cfg.rotate_limit)
    scale = 1.0 + rng.uniform(-cfg.scale_limit, cfg.scale_limit)
    tx = rng.uniform(-cfg.shift_limit, cfg.shift_limit) * w
    ty = rng.uniform(-cfg.shift_limit, cfg.shift_limit) * h
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REFLECT_101)
    mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return img, mask


def apply_augmentation(img: np.ndarray, mask: np.ndarray, cfg: AugConfig,
                       rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    if rng.random() < cfg.hflip:
        img, mask = img[:, ::-1], mask[:, ::-1]
    if rng.random() < cfg.vflip:
        img, mask = img[::-1], mask[::-1]
    if cfg.rot90 and rng.random() < cfg.rot90:
        k = int(rng.integers(1, 4))
        img, mask = np.rot90(img, k), np.rot90(mask, k)
    img = np.ascontiguousarray(img)
    mask = np.ascontiguousarray(mask)
    if rng.random() < cfg.affine_p:
        img, mask = _affine(img, mask, cfg, rng)
    if rng.random() < cfg.photometric_p:
        alpha = 1.0 + rng.uniform(-cfg.contrast_limit, cfg.contrast_limit)
        beta = rng.uniform(-cfg.brightness_limit, cfg.brightness_limit) * 255.0
        img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    if rng.random() < cfg.gamma_p:
        g = rng.uniform(*cfg.gamma_limit)
        lut = np.clip(((np.arange(256) / 255.0) ** g) * 255.0, 0, 255).astype(np.uint8)
        img = cv2.LUT(img, lut)
    return img, mask


# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------
class SegmentationDataset(Dataset):
    def __init__(
        self,
        samples: Sequence[Sample],
        img_size: int = 256,
        train: bool = False,
        in_channels: int = 3,
        aug: Optional[AugConfig] = None,
        seed: int = 0,
        cache_in_ram: bool = True,
    ) -> None:
        self.samples = list(samples)
        self.img_size = img_size
        self.train = train
        self.in_channels = in_channels
        self.aug = aug or AugConfig()
        self.seed = seed
        self.cache_in_ram = cache_in_ram
        self._cache: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}

    def __len__(self) -> int:
        return len(self.samples)

    def _load_resized(self, idx: int) -> Tuple[np.ndarray, np.ndarray]:
        if self.cache_in_ram and idx in self._cache:
            return self._cache[idx]
        s = self.samples[idx]
        img = read_image(s.image_path, in_channels=3)
        mask = read_mask_union(s.mask_paths, img.shape[:2])
        size = (self.img_size, self.img_size)
        img = cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)
        if self.cache_in_ram:
            self._cache[idx] = (img, mask)
        return img, mask

    def __getitem__(self, idx: int):
        img, mask = self._load_resized(idx)
        img, mask = img.copy(), mask.copy()
        if self.train:
            # Per-(epoch, index) RNG: reproducible yet not identical every epoch.
            rng = np.random.default_rng((self.seed * 1_000_003 + idx) % (2**31))
            img, mask = apply_augmentation(img, mask, self.aug, rng)

        x = img.astype(np.float32) / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
        y = torch.from_numpy(np.ascontiguousarray(mask.astype(np.float32)))[None]
        return {
            "image": x,
            "mask": y,
            "index": idx,
            "sample_id": self.samples[idx].sample_id,
            "group": self.samples[idx].group,
        }


def make_loader(ds: SegmentationDataset, batch_size: int, shuffle: bool,
                num_workers: int = 2, drop_last: bool = False) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last,
        persistent_workers=num_workers > 0,
    )


# --------------------------------------------------------------------------
# Splitting
# --------------------------------------------------------------------------
def _stable_hash(text: str) -> int:
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)


def stratified_folds(
    samples: Sequence[Sample], n_folds: int = 5, seed: int = 0
) -> List[np.ndarray]:
    """Stratified K-fold on the ``group`` label, with a deterministic fallback.

    Stratification matters more than usual here: BUSI's 'normal' class is 133 of
    780 images and is entirely empty masks, so an unstratified fold can end up
    with a wildly different empty-mask rate and a Dice that is not comparable
    across folds.
    """
    labels = [s.group or "all" for s in samples]
    idx = np.arange(len(samples))
    try:
        from sklearn.model_selection import StratifiedKFold
        uniq, counts = np.unique(labels, return_counts=True)
        if counts.min() >= n_folds:
            skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
            return [test for _, test in skf.split(idx, labels)]
    except Exception:
        pass
    rng = np.random.default_rng(seed)
    perm = rng.permutation(idx)
    return [np.sort(a) for a in np.array_split(perm, n_folds)]


def split_train_val(
    train_idx: np.ndarray, samples: Sequence[Sample], val_frac: float = 0.15, seed: int = 0
) -> Tuple[np.ndarray, np.ndarray]:
    labels = [samples[i].group or "all" for i in train_idx]
    try:
        from sklearn.model_selection import train_test_split
        tr, va = train_test_split(
            train_idx, test_size=val_frac, random_state=seed, stratify=labels
        )
        return np.sort(tr), np.sort(va)
    except Exception:
        rng = np.random.default_rng(seed)
        perm = rng.permutation(train_idx)
        k = max(1, int(round(val_frac * len(perm))))
        return np.sort(perm[k:]), np.sort(perm[:k])


def label_subset(
    train_idx: np.ndarray, samples: Sequence[Sample], frac: float, seed: int = 0
) -> np.ndarray:
    """Stratified subsample of the training indices for the low-label ablation."""
    if frac >= 1.0:
        return train_idx
    labels = np.array([samples[i].group or "all" for i in train_idx])
    rng = np.random.default_rng(seed)
    keep: List[int] = []
    for lab in np.unique(labels):
        pool = train_idx[labels == lab]
        k = max(1, int(round(frac * len(pool))))
        keep.extend(rng.choice(pool, size=k, replace=False).tolist())
    return np.sort(np.array(keep))


def summarize_samples(samples: Sequence[Sample]) -> Dict[str, object]:
    groups: Dict[str, int] = {}
    for s in samples:
        groups[s.group or "all"] = groups.get(s.group or "all", 0) + 1
    return {"n": len(samples), "groups": groups}

In [ ]:
# ===== sparcseg/data/busi.py =================================================
"""BUSI -- Breast Ultrasound Images (Al-Dhabyani et al., 2020).

Layout (the common Kaggle mirror is ``Dataset_BUSI_with_GT/``)::

    benign/     benign (1).png, benign (1)_mask.png, benign (1)_mask_1.png, ...
    malignant/  malignant (1).png, malignant (1)_mask.png, ...
    normal/     normal (1).png, normal (1)_mask.png        <- all-zero masks

Two traps handled here:
  1. multi-lesion cases ship extra ``_mask_N.png`` files -> unioned;
  2. the 133 'normal' cases have empty masks and must be *kept*, because a
     model whose reasoning state is decorative tends to hallucinate lesions
     exactly there.  They are stratified into every fold.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional, Sequence

pass  # (inlined below)
pass  # (inlined below)

HINTS_BUSI = ("busi", "dataset_busi", "breast-ultrasound", "breast_ultrasound", "breast")
_MASK_RE = re.compile(r"_mask(_\d+)?$", re.IGNORECASE)


def _is_busi_mask(p: Path) -> bool:
    return bool(_MASK_RE.search(p.stem)) and p.parent.name.lower() in {
        "benign", "malignant", "normal"
    }


def find_root_busi(root: Optional[str] = None) -> Path:
    if root:
        p = Path(root)
        if p.is_dir():
            return p
    found = find_by_signature(_is_busi_mask, hints=HINTS_BUSI, min_hits=20)
    if found is None:
        # Some mirrors flatten the class folders; retry without the folder check.
        found = find_by_signature(lambda p: bool(_MASK_RE.search(p.stem)),
                                  hints=HINTS_BUSI, min_hits=20)
    if found is None:
        raise DatasetNotFound("BUSI", HINTS_BUSI)
    return found


def build_index_busi(root: Optional[str] = None, keep_normal: bool = True) -> List[Sample]:
    base = find_root_busi(root)
    by_stem: Dict[Path, List[Path]] = {}
    images: Dict[Path, Path] = {}

    for f in walk_files(base):
        stem = f.stem
        if _MASK_RE.search(stem):
            key = f.parent / _MASK_RE.sub("", stem)
            by_stem.setdefault(key, []).append(f)
        else:
            images[f.parent / stem] = f

    samples: List[Sample] = []
    for key, img_path in sorted(images.items()):
        masks = sorted(by_stem.get(key, []))
        if not masks:
            continue  # an image with no annotation is unusable, not a negative
        group = img_path.parent.name.lower()
        if group not in {"benign", "malignant", "normal"}:
            group = "unknown"
        if group == "normal" and not keep_normal:
            continue
        samples.append(
            Sample(image_path=img_path, mask_paths=masks, group=group,
                   sample_id=f"{group}/{img_path.stem}")
        )

    if not samples:
        raise DatasetNotFound("BUSI (found a root but no image/mask pairs)", HINTS_BUSI)
    return sorted(samples, key=lambda s: s.sample_id)

In [ ]:
# ===== sparcseg/data/isic.py =================================================
"""ISIC 2018 Task 1 -- Skin Lesion Boundary Segmentation.

Layout varies by mirror; both of these are handled::

    ISIC2018_Task1-2_Training_Input/ISIC_0000000.jpg
    ISIC2018_Task1_Training_GroundTruth/ISIC_0000000_segmentation.png

    images/ISIC_0000000.jpg   +   masks/ISIC_0000000_segmentation.png

Official Training / Validation / Test splits are recorded in ``split_hint`` when
the directory names reveal them.  We nonetheless run our own stratified 5-fold
over the *training* pool for the main table, because the official validation set
is only 100 images -- far too small to support the paired tests this paper needs.
Lesion-area quintile is used as the stratification label: ISIC has no class
label, and area is the variable that most strongly predicts Dice.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional

import cv2
import numpy as np

pass  # (inlined below)
pass  # (inlined below)

HINTS_ISIC = ("isic2018", "isic-2018", "isic_2018", "isic", "skin")


def _is_isic_mask(p: Path) -> bool:
    return p.stem.lower().endswith("_segmentation")


def find_root_isic(root: Optional[str] = None) -> Path:
    if root:
        p = Path(root)
        if p.is_dir():
            return p
    found = find_by_signature(_is_isic_mask, hints=HINTS_ISIC, min_hits=20)
    if found is None:
        raise DatasetNotFound("ISIC 2018 Task 1", HINTS_ISIC)
    # find_by_signature lands on the GroundTruth folder; step up so that the
    # sibling Input folder is inside the search scope.
    return found.parent if found.parent.is_dir() else found


def _split_hint_isic(path: Path) -> str:
    low = str(path).lower()
    if "valid" in low:
        return "val"
    if "test" in low:
        return "test"
    return "train"


def build_index_isic(root: Optional[str] = None, area_bins: int = 5) -> List[Sample]:
    base = find_root_isic(root)
    masks: Dict[str, Path] = {}
    images: Dict[str, Path] = {}

    for f in walk_files(base):
        stem = f.stem
        if _is_isic_mask(f):
            masks[stem[: -len("_segmentation")]] = f
        elif stem.upper().startswith("ISIC"):
            images.setdefault(stem, f)

    samples: List[Sample] = []
    for key, img in sorted(images.items()):
        m = masks.get(key)
        if m is None:
            continue
        samples.append(
            Sample(image_path=img, mask_paths=[m], group="",
                   split_hint=_split_hint_isic(img), sample_id=key)
        )
    if not samples:
        raise DatasetNotFound("ISIC 2018 (found a root but no image/mask pairs)", HINTS_ISIC)

    _assign_area_groups(samples, area_bins)
    return sorted(samples, key=lambda s: s.sample_id)


def _assign_area_groups(samples: List[Sample], bins: int) -> None:
    """Stratify by lesion-area quantile, read from a downsampled mask (cheap)."""
    areas = np.zeros(len(samples), dtype=np.float32)
    for i, s in enumerate(samples):
        m = cv2.imread(str(s.mask_paths[0]), cv2.IMREAD_GRAYSCALE)
        if m is None:
            areas[i] = np.nan
            continue
        small = cv2.resize(m, (64, 64), interpolation=cv2.INTER_NEAREST)
        areas[i] = float((small > 127).mean())
    finite = areas[np.isfinite(areas)]
    if finite.size == 0:
        return
    edges = np.quantile(finite, np.linspace(0, 1, bins + 1)[1:-1])
    for i, s in enumerate(samples):
        a = areas[i]
        s.group = "area_nan" if not np.isfinite(a) else f"area_q{int(np.searchsorted(edges, a))}"

In [ ]:
# ===== sparcseg/data/brisc.py ================================================
"""BRISC 2025 -- Brain Tumor MRI classification + segmentation.

Segmentation layout::

    segmentation_task/train/images/*.jpg
    segmentation_task/train/masks/*.png
    segmentation_task/test/images, .../masks

Filenames usually encode the tumour subtype (glioma / meningioma / pituitary /
no_tumor); that string becomes the stratification group.  Tumour-free slices
carry an all-zero mask and are kept for the same reason as BUSI 'normal'.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional

pass  # (inlined below)
pass  # (inlined below)

HINTS_BRISC = ("brisc", "brisc2025", "brisc-2025", "brain-tumor", "brain_tumor", "brain")
SUBTYPES_BRISC = ("glioma", "meningioma", "pituitary", "notumor", "no_tumor", "normal")


def _in_masks_dir(p: Path) -> bool:
    return p.parent.name.lower() in {"masks", "mask", "labels", "groundtruth", "gt"}


def find_root_brisc(root: Optional[str] = None) -> Path:
    if root:
        p = Path(root)
        if p.is_dir():
            return p
    found = find_by_signature(_in_masks_dir, hints=HINTS_BRISC, min_hits=20)
    if found is None:
        raise DatasetNotFound("BRISC 2025", HINTS_BRISC)
    return found


def _subtype_brisc(name: str) -> str:
    low = name.lower()
    for s in SUBTYPES_BRISC:
        if s in low:
            return "notumor" if s in {"no_tumor", "normal"} else s
    return "unknown"


def _split_hint_brisc(path: Path) -> str:
    low = str(path).lower()
    if "test" in low:
        return "test"
    if "val" in low:
        return "val"
    return "train"


def _normalize_stem_brisc(stem: str) -> str:
    """Strip the '_mask' / '_seg' suffix some mirrors add to mask filenames."""
    return re.sub(r"[_-](mask|seg|segmentation|label)$", "", stem, flags=re.IGNORECASE)


def build_index_brisc(root: Optional[str] = None) -> List[Sample]:
    base = find_root_brisc(root)
    # Walk from the segmentation task root so images/ is in scope too.
    search_root = base
    for _ in range(3):
        if any(d.name.lower() in {"images", "image"} for d in search_root.iterdir() if d.is_dir()):
            break
        if search_root.parent == search_root:
            break
        search_root = search_root.parent

    masks: Dict[str, Path] = {}
    images: Dict[str, Path] = {}
    for f in walk_files(search_root):
        key_split = _split_hint_brisc(f)
        stem = _normalize_stem_brisc(f.stem)
        key = f"{key_split}/{stem}"
        if _in_masks_dir(f):
            masks[key] = f
        elif f.parent.name.lower() in {"images", "image"}:
            images.setdefault(key, f)

    samples: List[Sample] = []
    for key, img in sorted(images.items()):
        m = masks.get(key)
        if m is None:
            continue
        split, stem = key.split("/", 1)
        samples.append(
            Sample(image_path=img, mask_paths=[m], group=_subtype_brisc(stem),
                   split_hint=split, sample_id=key)
        )
    if not samples:
        raise DatasetNotFound("BRISC 2025 (found a root but no image/mask pairs)", HINTS_BRISC)
    return sorted(samples, key=lambda s: s.sample_id)

---
## Part 2 - Model

All five methods share this backbone with identical initialisation and identical parameter count in the perceptual path. That is the only way the comparison isolates the reasoning mechanism rather than backbone capacity.

In [ ]:
# ===== sparcseg/models/backbone.py ===========================================
"""Shared encoder/decoder backbone.

Every method in the paper -- single-shot, dense-unrolled, PTEA-lite and
SPARC-Seg -- is built on *this* module with identical weights-initialisation and
identical parameter count in the perceptual path.  That is the only way the
comparison isolates the reasoning mechanism rather than backbone capacity.

The encoder produces the "raw evidence" g_phi(x): a stride-4 feature map with
``sketch_dim`` channels.  The reasoning loop then operates entirely at stride 4
(64x64 for a 256x256 input), which is what makes K unrolled steps affordable on
a single T4.
"""

from __future__ import annotations

import os
import warnings
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------
# Offline-friendly pretrained weight loading
# --------------------------------------------------------------------------
def _find_local_resnet_weights(arch: str = "resnet34") -> Optional[Path]:
    """Look for ImageNet weights already on disk.

    Kaggle notebooks frequently run with internet disabled.  Several widely
    mirrored Kaggle datasets ship torchvision checkpoints; we scan for them so
    the notebook does not silently fall back to random initialisation (which
    would cost ~4-6 Dice points and make the numbers non-comparable to the
    literature).
    """
    patterns = [f"{arch}-", f"{arch}_", arch]
    roots = [
        Path(torch.hub.get_dir()) / "checkpoints",
        Path.home() / ".cache/torch/hub/checkpoints",
        Path("/kaggle/input"),
    ]
    for root in roots:
        if not root.is_dir():
            continue
        try:
            for dirpath, _dirnames, filenames in os.walk(root):
                for fn in filenames:
                    low = fn.lower()
                    if low.endswith((".pth", ".pt")) and any(p in low for p in patterns):
                        return Path(dirpath) / fn
        except Exception:
            continue
    return None


def build_resnet(arch: str = "resnet34", pretrained: bool = True) -> Tuple[nn.Module, List[int]]:
    import torchvision

    ctor = getattr(torchvision.models, arch)
    channels = {
        "resnet18": [64, 64, 128, 256, 512],
        "resnet34": [64, 64, 128, 256, 512],
        "resnet50": [64, 256, 512, 1024, 2048],
    }[arch]

    net = None
    if pretrained:
        try:
            weights_enum = getattr(torchvision.models, f"{arch.capitalize()}_Weights", None)
            if weights_enum is None:
                weights_enum = getattr(
                    torchvision.models, f"ResNet{arch.replace('resnet','')}_Weights"
                )
            net = ctor(weights=weights_enum.IMAGENET1K_V1)
            print(f"[backbone] loaded torchvision ImageNet weights for {arch}")
        except Exception as e:
            local = _find_local_resnet_weights(arch)
            if local is not None:
                net = ctor(weights=None)
                try:
                    sd = torch.load(local, map_location="cpu", weights_only=True)
                except TypeError:
                    sd = torch.load(local, map_location="cpu")
                if isinstance(sd, dict) and "state_dict" in sd:
                    sd = sd["state_dict"]
                missing, unexpected = net.load_state_dict(sd, strict=False)
                print(f"[backbone] loaded local ImageNet weights from {local} "
                      f"(missing={len(missing)}, unexpected={len(unexpected)})")
            else:
                warnings.warn(
                    f"\n{'!' * 70}\n"
                    f"Could not obtain ImageNet weights for {arch} ({type(e).__name__}).\n"
                    f"Falling back to RANDOM INITIALISATION. Absolute Dice will be\n"
                    f"several points below published numbers. Turn Kaggle internet ON\n"
                    f"(Settings -> Internet) or attach a torchvision-weights dataset.\n"
                    f"{'!' * 70}",
                    RuntimeWarning,
                )
                net = ctor(weights=None)
    if net is None:
        net = ctor(weights=None)
    return net, channels


class ResNetEncoder(nn.Module):
    """ResNet trunk exposing the five standard pyramid levels."""

    def __init__(self, arch: str = "resnet34", pretrained: bool = True,
                 in_channels: int = 3) -> None:
        super().__init__()
        net, self.out_channels = build_resnet(arch, pretrained)
        if in_channels != 3:
            old = net.conv1
            new = nn.Conv2d(in_channels, old.out_channels, old.kernel_size,
                            old.stride, old.padding, bias=False)
            with torch.no_grad():
                w = old.weight.mean(dim=1, keepdim=True).repeat(1, in_channels, 1, 1)
                new.weight.copy_(w * (3.0 / in_channels))
            net.conv1 = new
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu)
        self.pool = net.maxpool
        self.layer1, self.layer2 = net.layer1, net.layer2
        self.layer3, self.layer4 = net.layer3, net.layer4

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        f0 = self.stem(x)                 # stride  2
        f1 = self.layer1(self.pool(f0))   # stride  4
        f2 = self.layer2(f1)              # stride  8
        f3 = self.layer3(f2)              # stride 16
        f4 = self.layer4(f3)              # stride 32
        return [f0, f1, f2, f3, f4]


def conv_block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class UNetDecoderToStride4(nn.Module):
    """Decode the pyramid down to a stride-4 map with ``out_dim`` channels.

    This map is the working sketch S_0 = g_phi(x): the raw image evidence that
    the reasoning loop then revises.
    """

    def __init__(self, enc_channels: Sequence[int], out_dim: int = 64,
                 width: int = 128) -> None:
        super().__init__()
        c0, c1, c2, c3, c4 = enc_channels
        self.up3 = conv_block(c4 + c3, width)
        self.up2 = conv_block(width + c2, width)
        self.up1 = conv_block(width + c1, width)
        self.project = nn.Sequential(
            nn.Conv2d(width, out_dim, 1, bias=False),
            nn.BatchNorm2d(out_dim),
        )

    @staticmethod
    def _up_cat(x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([x, skip], dim=1)

    def forward(self, feats: Sequence[torch.Tensor]) -> torch.Tensor:
        f0, f1, f2, f3, f4 = feats
        x = self.up3(self._up_cat(f4, f3))
        x = self.up2(self._up_cat(x, f2))
        x = self.up1(self._up_cat(x, f1))
        return self.project(x)


class ReadoutHead(nn.Module):
    """S -> mask logits at full resolution.

    Kept deliberately shallow (two 3x3 convs and two upsamples): if the readout
    were a deep network it could repair an uninformative sketch, and the causal
    claims about S would lose their force.
    """

    def __init__(self, in_dim: int = 64, width: int = 32, n_classes: int = 1,
                 scale: int = 4) -> None:
        super().__init__()
        self.scale = scale
        self.body = nn.Sequential(
            nn.Conv2d(in_dim, width, 3, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            nn.Conv2d(width, width, 3, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(width, n_classes, 1)

    def forward(self, s: torch.Tensor,
                out_size: Optional[Tuple[int, int]] = None) -> torch.Tensor:
        x = self.body(s)
        logits = self.classifier(x)
        if out_size is None:
            out_size = (logits.shape[-2] * self.scale, logits.shape[-1] * self.scale)
        return F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)

    def logits_at_sketch_res(self, s: torch.Tensor) -> torch.Tensor:
        """Mask logits at stride-4 resolution, used inside the energy so the
        shape prior does not pay for a 16x upsample at every descent step."""
        return self.classifier(self.body(s))


class Encoder(nn.Module):
    """Convenience wrapper: image -> evidence sketch g_phi(x)."""

    def __init__(self, arch: str = "resnet34", pretrained: bool = True,
                 sketch_dim: int = 64, in_channels: int = 3, width: int = 128) -> None:
        super().__init__()
        self.trunk = ResNetEncoder(arch, pretrained, in_channels)
        self.decoder = UNetDecoderToStride4(self.trunk.out_channels, sketch_dim, width)
        self.sketch_dim = sketch_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.trunk(x))

In [ ]:
# ===== sparcseg/models/dictionary.py =========================================
"""The concept dictionary and its proximal operators.

Design note -- a correction to the original proposal
----------------------------------------------------
The proposal wrote the sketch as S in R^{HxWxd} but the code as z in R^m, i.e.
a single global vector.  Those two are dimensionally incompatible inside
||S - Dz||_F.  The code here is *spatial*: z in R^{B x m x h x w}, with D applied
as a 1x1 convolution, so ``Dz`` reconstructs a sketch of the right shape.

That fix creates a second problem which the proposal's audit story depends on:
a purely elementwise L1 penalty gives a code that is sparse *per pixel* while
still using all m atoms somewhere in the image, so "at step 2 the model used
atoms #14 and #31" would be false.  We therefore use a sparse-group-lasso
penalty (Friedman, Hastie & Tibshirani, 2010):

    lambda_1 * ||z||_1  +  lambda_g * sum_j ||z_{.,j,.,.}||_2

The group term is taken over each atom's entire spatial map, so it drives whole
atoms to zero image-wide.  Its proximal operator is exact and cheap: elementwise
soft-threshold followed by block soft-threshold.  Non-negativity (optional, on
by default) makes a coefficient read as "how present is this concept", which is
what the interpretability claim needs.
"""

from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def soft_threshold(v: torch.Tensor, t: float | torch.Tensor,
                   nonneg: bool = True) -> torch.Tensor:
    """prox of t * ||.||_1, optionally composed with the non-negative orthant."""
    if nonneg:
        return F.relu(v - t)
    return torch.sign(v) * F.relu(v.abs() - t)


def block_soft_threshold(z: torch.Tensor, t: float | torch.Tensor,
                         eps: float = 1e-8) -> torch.Tensor:
    """prox of t * sum_j ||z_j||_2 with groups = (batch, atom) spatial maps."""
    if (isinstance(t, float) and t <= 0.0):
        return z
    norms = z.flatten(2).norm(dim=2).clamp_min(eps)          # (B, m)
    scale = F.relu(1.0 - t / norms)[:, :, None, None]
    return z * scale


def prox_sparse_group(z: torch.Tensor, t_l1: float, t_group: float,
                      nonneg: bool = True) -> torch.Tensor:
    """Exact prox of  t_l1*||z||_1 + t_group*sum_j||z_j||_2  (+ nonnegativity).

    Composition order is not arbitrary: Friedman et al. show the sparse-group
    prox factorises as block-threshold(soft-threshold(.)).
    """
    return block_soft_threshold(soft_threshold(z, t_l1, nonneg=nonneg), t_group)


def project_topk_groups(z: torch.Tensor, k: int, nonneg: bool = True,
                        straight_through: bool = False) -> torch.Tensor:
    """Projection onto {at most k atoms active image-wide} (intersected with the
    non-negative orthant when ``nonneg``).

    Why this exists, and why it is the default
    ------------------------------------------
    An L1 penalty controls sparsity only *relative to the scale of the data it
    is applied to*.  During training the sketch S and the dictionary both change
    scale, and the effective threshold eta*lambda_1 (eta = 1/||D^T D||_2) drifts
    with them.  Measured on a trained model, a fixed lambda_1 that gave ~4 active
    atoms at initialisation gave ~47 of 48 after training -- i.e. the sparsity,
    and with it the whole audit story, silently evaporated.

    Hard group-sparsity is scale-free: k atoms are active by construction, on
    every dataset, before and after training, with no per-dataset tuning.  The
    descent guarantee survives intact -- for any *closed* set C (convex or not)
    and a step size eta <= 1/L, the projected-gradient step
    ``z+ = P_C(z - eta grad f(z))`` satisfies f(z+) <= f(z) whenever z is in C,
    because P_C minimises the same quadratic majorant that the proximal step
    minimises.  This is exactly the Iterative Hard Thresholding argument
    (Blumensath & Davies, 2009), applied here at group rather than element
    granularity.

    Ties are resolved by keeping every atom at the threshold norm, so the active
    count can exceed k by the size of a tie -- in practice never more than one.

    ``straight_through`` (training only) keeps the forward value exactly equal to
    the hard projection -- so every energy the loop evaluates is still evaluated
    at a feasible point and the descent guarantee is untouched -- while letting
    gradient reach the atoms the mask zeroed (Bengio et al., 2013).  Without it a
    hard gate is absorbing: once an atom stops being selected it receives no
    gradient and can never be selected again.  Measured on a trained model, the
    pure-hard variant left 32 of 48 atoms permanently dead.
    """
    if nonneg:
        z = F.relu(z)
    m = z.shape[1]
    if k >= m:
        return z
    norms = z.flatten(2).norm(dim=2)                     # (B, m)
    kth = torch.topk(norms, k, dim=1).values[:, -1:]     # (B, 1)
    keep = (norms >= kth).to(z.dtype)[:, :, None, None]
    z_hard = z * keep
    if straight_through and z.requires_grad:
        return z + (z_hard - z).detach()
    return z_hard


class ConceptDictionary(nn.Module):
    """Overcomplete dictionary D in R^{d x m}, applied convolutionally.

    Atoms are renormalised to unit L2 norm after every optimiser step.  This is
    not cosmetic: without it the network can defeat the L1 penalty by inflating
    atom norms and shrinking coefficients, which would make the sparsity level --
    and therefore every faithfulness number -- meaningless.
    """

    def __init__(self, sketch_dim: int = 64, dict_size: int = 192,
                 init: str = "orthogonal") -> None:
        super().__init__()
        self.d = sketch_dim
        self.m = dict_size
        w = torch.empty(sketch_dim, dict_size)
        if init == "orthogonal" and dict_size >= sketch_dim:
            nn.init.orthogonal_(w)
        else:
            nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        self.weight = nn.Parameter(w)               # (d, m)
        self.register_buffer("_lipschitz", torch.tensor(1.0))
        self.register_buffer("_u", F.normalize(torch.randn(dict_size), dim=0))
        self.normalize_atoms()

    # -- linear operators ---------------------------------------------------
    @property
    def D(self) -> torch.Tensor:
        return self.weight

    def synthesize(self, z: torch.Tensor) -> torch.Tensor:
        """D z : (B, m, h, w) -> (B, d, h, w)."""
        return F.conv2d(z, self.weight[:, :, None, None])

    def analyze(self, s: torch.Tensor) -> torch.Tensor:
        """D^T s : (B, d, h, w) -> (B, m, h, w)."""
        return F.conv2d(s, self.weight.t()[:, :, None, None])

    # -- housekeeping -------------------------------------------------------
    @torch.no_grad()
    def normalize_atoms(self) -> None:
        self.weight.data = F.normalize(self.weight.data, dim=0, eps=1e-8)

    @torch.no_grad()
    def lipschitz(self, n_iter: int = 20, refresh: bool = True) -> torch.Tensor:
        """L = ||D^T D||_2 = sigma_max(D)^2, by power iteration.

        The z-step uses eta = 1/L, which is exactly the condition under which
        the proximal-gradient descent lemma guarantees E(z_{t+1}) <= E(z_t).
        """
        if not refresh:
            return self._lipschitz
        u = self._u
        Dm = self.weight
        for _ in range(n_iter):
            v = Dm @ u
            u = Dm.t() @ v
            u = F.normalize(u, dim=0, eps=1e-8)
        sigma_sq = (Dm @ u).pow(2).sum() / (u.pow(2).sum() + 1e-12)
        self._u.copy_(u)
        self._lipschitz.copy_(sigma_sq.clamp_min(1e-6))
        return self._lipschitz

    # -- diagnostics --------------------------------------------------------
    @torch.no_grad()
    def coherence(self) -> torch.Tensor:
        """Max off-diagonal |<d_i, d_j>|. High coherence means two 'concepts'
        are near-duplicates, which would inflate the sufficiency score for
        trivial reasons -- so it is reported alongside the faithfulness table."""
        G = (self.weight.t() @ self.weight).abs()
        G.fill_diagonal_(0.0)
        return G.max()

    @torch.no_grad()
    def usage_stats(self, z: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Per-atom activation mass and the image-wide active-atom count."""
        mass = z.flatten(2).abs().sum(dim=2)                  # (B, m)
        active = (mass > 1e-6).float().sum(dim=1)             # (B,)
        per_pixel_active = (z.abs() > 1e-6).float().sum(dim=1).flatten(1).mean(dim=1)
        return {
            "atom_mass": mass,
            "atoms_active_per_image": active,
            "atoms_active_per_pixel": per_pixel_active,
        }


class CodePredictor(nn.Module):
    """Optional learned initialisation z_0 = ReLU(W s) (LISTA-style).

    Gregor & LeCun showed a learned initialisation reaches a given sparse-coding
    accuracy in far fewer iterations.  Here it matters for a second reason: it
    lets K stay small enough that K unrolled steps fit in a T4's memory.
    """

    def __init__(self, sketch_dim: int, dict_size: int, nonneg: bool = True) -> None:
        super().__init__()
        self.proj = nn.Conv2d(sketch_dim, dict_size, 1, bias=True)
        self.nonneg = nonneg
        nn.init.zeros_(self.proj.bias)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        z = self.proj(s)
        return F.relu(z) if self.nonneg else z

In [ ]:
# ===== sparcseg/models/energy.py =============================================
"""The energy functional E(z, S) and its descent machinery.

Correction to the original proposal
-----------------------------------
The proposal put a persistent-homology penalty inside E and then invoked
proximal-descent theory.  Those two do not fit together: a PH loss is
piecewise-linear in the filtration values, so its gradient is not
Lipschitz-continuous and no finite L exists -- the descent lemma's hypothesis
simply fails, and the stated Proposition would be false as written.  (It is also
far too slow to evaluate inside an unrolled loop on a T4.)

The fix keeps the guarantee honest and costs nothing scientifically:

  * inside E, the shape prior is a *smooth* surrogate -- Huber-smoothed total
    variation (a differentiable perimeter/boundary-length term) plus a Laplacian
    curvature term.  Both have Lipschitz-continuous gradients on bounded sets,
    so the descent lemma genuinely applies;
  * persistent-homology-flavoured structure is reported as an *evaluation*
    metric (Betti-0/1 error in ``metrics.py``), where non-differentiability is
    irrelevant.

Additionally, rather than claiming an analytic Lipschitz constant for the
composite S-step, evaluation uses Armijo backtracking.  Backtracking gives
monotone descent for *any* term with a finite local Lipschitz constant without
needing to know its value -- a strictly stronger and more defensible claim than
asserting a hand-derived L'.  ``EnergyTrace.is_monotone`` verifies this
numerically on every run, so the paper can report the guarantee as *audited*,
not merely asserted.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------
# Smooth shape prior
# --------------------------------------------------------------------------
def huber_tv(p: torch.Tensor, delta: float = 0.05) -> torch.Tensor:
    """Huber-smoothed total variation of a probability map.

    Plain TV (|grad p|) is non-differentiable at zero; the Huber form is C^1 with
    gradient Lipschitz constant bounded by 8/delta on a 4-neighbourhood grid,
    which is exactly the property the descent lemma needs.
    """
    dx = p[..., :, 1:] - p[..., :, :-1]
    dy = p[..., 1:, :] - p[..., :-1, :]
    def _h(v: torch.Tensor) -> torch.Tensor:
        a = v.abs()
        return torch.where(a <= delta, 0.5 * v.pow(2) / delta, a - 0.5 * delta)
    return _h(dx).flatten(1).sum(1) + _h(dy).flatten(1).sum(1)


def laplacian_energy(p: torch.Tensor) -> torch.Tensor:
    """||Laplacian(p)||^2 -- penalises jagged, high-curvature boundaries.

    Quadratic, hence gradient-Lipschitz with a constant equal to twice the
    squared spectral norm of the Laplacian stencil (<= 128 on this grid).
    """
    k = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]],
                     device=p.device, dtype=p.dtype).view(1, 1, 3, 3)
    c = p.shape[1]
    lap = F.conv2d(F.pad(p, (1, 1, 1, 1), mode="replicate"), k.expand(c, 1, 3, 3), groups=c)
    return lap.pow(2).flatten(1).sum(1)


@dataclass
class ShapePriorConfig:
    tv_weight: float = 1.0
    curvature_weight: float = 0.25
    huber_delta: float = 0.05
    normalize: bool = True     # divide by pixel count so lambda_topo is size-free


class SmoothShapePrior(nn.Module):
    """R_topo(S): anatomical-plausibility term, evaluated on the stride-4 soft mask."""

    def __init__(self, readout: nn.Module, cfg: Optional[ShapePriorConfig] = None) -> None:
        super().__init__()
        self.readout = readout          # shared with the model; NOT a copy
        self.cfg = cfg or ShapePriorConfig()

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        p = torch.sigmoid(self.readout.logits_at_sketch_res(s))
        val = self.cfg.tv_weight * huber_tv(p, self.cfg.huber_delta)
        val = val + self.cfg.curvature_weight * laplacian_energy(p)
        if self.cfg.normalize:
            val = val / float(p.shape[-1] * p.shape[-2])
        return val                      # (B,)


# --------------------------------------------------------------------------
# Energy functional
# --------------------------------------------------------------------------
@dataclass
class EnergyWeights:
    lambda_l1: float = 0.05
    lambda_group: float = 0.02
    lambda_topo: float = 0.10
    lambda_evidence: float = 0.50


class EnergyFunctional(nn.Module):
    """E(z, S) = 1/2||S - Dz||^2 + l1*||z||_1 + lg*sum_j||z_j||_2
                 + l_topo*R(S) + l_evid*||S - g(x)||^2

    All terms are returned per-sample so that per-image energy traces can be
    correlated against per-image Dice -- the diagnostic that answers the
    "your energy decreases, but does the *answer* improve?" question.
    """

    def __init__(self, dictionary: nn.Module, shape_prior: nn.Module,
                 weights: Optional[EnergyWeights] = None) -> None:
        super().__init__()
        self.dict = dictionary
        self.prior = shape_prior
        self.w = weights or EnergyWeights()

    # -- individual terms ---------------------------------------------------
    def recon_term(self, z: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        return 0.5 * (s - self.dict.synthesize(z)).pow(2).flatten(1).sum(1)

    def sparsity_term(self, z: torch.Tensor) -> torch.Tensor:
        l1 = z.abs().flatten(1).sum(1)
        grp = z.flatten(2).norm(dim=2).sum(1)
        return self.w.lambda_l1 * l1 + self.w.lambda_group * grp

    def evidence_term(self, s: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        return (s - g).pow(2).flatten(1).sum(1)

    # -- composites ---------------------------------------------------------
    def smooth_in_s(self, z: torch.Tensor, s: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        """The part of E that is differentiable in S (everything but ||z||_1)."""
        return (
            self.recon_term(z, s)
            + self.w.lambda_topo * self.prior(s)
            + self.w.lambda_evidence * self.evidence_term(s, g)
        )

    def total(self, z: torch.Tensor, s: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        return self.smooth_in_s(z, s, g) + self.sparsity_term(z)

    def breakdown(self, z: torch.Tensor, s: torch.Tensor,
                  g: torch.Tensor) -> Dict[str, torch.Tensor]:
        return {
            "recon": self.recon_term(z, s),
            "sparsity": self.sparsity_term(z),
            "topo": self.w.lambda_topo * self.prior(s),
            "evidence": self.w.lambda_evidence * self.evidence_term(s, g),
            "total": self.total(z, s, g),
        }

    # -- gradients ----------------------------------------------------------
    def grad_s(self, z: torch.Tensor, s: torch.Tensor, g: torch.Tensor,
               create_graph: bool = False) -> torch.Tensor:
        """dE/dS of the smooth part.

        The reconstruction and evidence gradients are written in closed form;
        only the shape prior goes through autograd.  This keeps the unrolled
        graph small enough to backprop through K steps at batch size 8 on a T4.
        """
        with torch.enable_grad():
            s_req = s if (s.requires_grad and create_graph) else s.detach().requires_grad_(True)
            prior_val = self.prior(s_req).sum()
            (grad_prior,) = torch.autograd.grad(
                prior_val, s_req, create_graph=create_graph, retain_graph=create_graph
            )
        if not create_graph:
            grad_prior = grad_prior.detach()
        closed = (s - self.dict.synthesize(z)) + 2.0 * self.w.lambda_evidence * (s - g)
        return closed + self.w.lambda_topo * grad_prior

    def grad_z_smooth(self, z: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        """dE_smooth/dz = D^T (Dz - S)."""
        return self.dict.analyze(self.dict.synthesize(z) - s)


# --------------------------------------------------------------------------
# Line search
# --------------------------------------------------------------------------
def armijo_backtrack(
    objective: Callable[[torch.Tensor], torch.Tensor],
    s: torch.Tensor,
    grad: torch.Tensor,
    step0: torch.Tensor | float,
    shrink: float = 0.5,
    max_iter: int = 8,
    c: float = 1e-4,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Per-sample Armijo backtracking on the S-step.

    Returns (s_new, accepted_step).  Step sizes are per-sample tensors so one
    hard image in a batch cannot force a tiny step on every other image -- which
    in practice is the difference between the loop converging in 3 steps and
    stalling at 8.
    """
    with torch.no_grad():
        f0 = objective(s)                                   # (B,)
        gnorm2 = grad.pow(2).flatten(1).sum(1)              # (B,)
        if not torch.is_tensor(step0):
            step0 = torch.full_like(f0, float(step0))
        step = step0.clone()
        s_new = s - step[:, None, None, None] * grad
        for _ in range(max_iter):
            f_new = objective(s_new)
            ok = f_new <= f0 - c * step * gnorm2
            if bool(ok.all()):
                break
            step = torch.where(ok, step, step * shrink)
            s_new = s - step[:, None, None, None] * grad
        # Any sample still failing Armijo keeps its old S: descent is never
        # violated, at worst a step is skipped.
        f_new = objective(s_new)
        keep = (f_new <= f0)[:, None, None, None]
        s_new = torch.where(keep, s_new, s)
    return s_new, step


# --------------------------------------------------------------------------
# Trace bookkeeping
# --------------------------------------------------------------------------
@dataclass
class EnergyTrace:
    """Per-step energies, (K+1, B), plus the audit of the descent property."""
    values: List[torch.Tensor] = field(default_factory=list)
    steps_taken: Optional[torch.Tensor] = None

    def append(self, e: torch.Tensor) -> None:
        self.values.append(e.detach().float().cpu())

    def stack(self) -> torch.Tensor:
        return torch.stack(self.values, dim=0) if self.values else torch.empty(0)

    def deltas(self) -> torch.Tensor:
        v = self.stack()
        return v[1:] - v[:-1] if v.numel() else v

    def is_monotone(self, tol: float = 1e-5) -> torch.Tensor:
        """Per-sample flag: did E never increase along the trajectory?

        Reported in the paper as 'monotone-descent rate', which should be 1.00
        by construction; anything less is a bug and this is how we would see it.
        """
        d = self.deltas()
        if d.numel() == 0:
            return torch.ones(0, dtype=torch.bool)
        return (d <= tol).all(dim=0)

    def relative_drop(self) -> torch.Tensor:
        v = self.stack()
        if v.shape[0] < 2:
            return torch.zeros(v.shape[-1] if v.numel() else 0)
        return (v[0] - v[-1]) / (v[0].abs() + 1e-8)

In [ ]:
# ===== sparcseg/models/sparcseg.py ===========================================
"""SPARC-Seg: the reasoning model itself.

    image --> encoder --> evidence g(x) == S_0
                             |
                             v
              K steps of block-coordinate proximal descent on E(z, S)
                 z-step: prox_{eta*lambda}( z - eta * D^T (Dz - S) )
                 S-step: S - eta' * grad_S [ smooth part of E ]
                             |
                             v
                       readout(S_K) --> mask

The same class implements the *dense control* baseline (``sparse=False``):
identical architecture, identical parameter count, identical number of steps,
with lambda_1 = lambda_group = 0 and the prox replaced by the identity.  Sharing
one class is deliberate -- a separately written baseline is where accidental
asymmetries creep in, and this paper's central claim is a comparison between
these two configurations.

Intervention support is built into ``forward`` rather than bolted on afterwards:
``intervene`` is called at every step with the freshly computed code, so
necessity / sufficiency / transplant / steering all run through the model's real
inference path, not a re-implementation of it.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)

# A hook receives (step_index, z) and returns a possibly modified z.
Intervention = Callable[[int, torch.Tensor], torch.Tensor]


@dataclass
class ReasoningOutput:
    logits: torch.Tensor                      # (B, 1, H, W) final prediction
    logits_per_step: List[torch.Tensor] = field(default_factory=list)
    codes: List[torch.Tensor] = field(default_factory=list)     # z_1..z_K
    sketches: List[torch.Tensor] = field(default_factory=list)  # S_0..S_K
    energy: Optional[EnergyTrace] = None
    steps_used: Optional[torch.Tensor] = None                   # (B,) float
    evidence: Optional[torch.Tensor] = None                     # g(x) == S_0

    def detach(self) -> "ReasoningOutput":
        return ReasoningOutput(
            logits=self.logits.detach(),
            logits_per_step=[t.detach() for t in self.logits_per_step],
            codes=[t.detach() for t in self.codes],
            sketches=[t.detach() for t in self.sketches],
            energy=self.energy,
            steps_used=self.steps_used,
            evidence=None if self.evidence is None else self.evidence.detach(),
        )


class SPARCSeg(nn.Module):
    def __init__(
        self,
        sketch_dim: int = 64,
        dict_size: int = 192,
        n_steps: int = 4,
        lambda_l1: float = 0.05,
        lambda_group: float = 0.02,
        lambda_topo: float = 0.10,
        lambda_evidence: float = 0.50,
        s_step_init: float = 0.50,
        nonneg_code: bool = True,
        sparse: bool = True,
        sparsity_mode: str = "topk",
        topk_atoms: int = 8,
        straight_through: bool = True,
        backbone: str = "resnet34",
        pretrained: bool = True,
        in_channels: int = 3,
        n_classes: int = 1,
        learned_code_init: bool = True,
        shape_prior: Optional[ShapePriorConfig] = None,
    ) -> None:
        super().__init__()
        self.sparse = sparse
        self.n_steps = n_steps
        self.nonneg_code = nonneg_code
        self.sparsity_mode = sparsity_mode
        self.topk_atoms = topk_atoms
        self.straight_through = straight_through

        self.encoder = Encoder(backbone, pretrained, sketch_dim, in_channels)
        self.readout = ReadoutHead(sketch_dim, n_classes=n_classes, scale=4)
        self.dictionary = ConceptDictionary(sketch_dim, dict_size)
        self.code_init = CodePredictor(sketch_dim, dict_size, nonneg_code) if learned_code_init else None

        # In top-k mode the sparsity constraint is a SET, not a penalty: its
        # contribution to E is the indicator of the feasible set, which is 0 on
        # every iterate the loop ever visits.  So E is numerically identical to
        # the dense control's energy, and the two configurations differ *only*
        # in the feasible set the z-step projects onto.  That is the tightest
        # possible version of this paper's central comparison -- same energy,
        # same architecture, same parameters, same K; structure or no structure.
        use_penalty = sparse and sparsity_mode == "l1"
        weights = EnergyWeights(
            lambda_l1=lambda_l1 if use_penalty else 0.0,
            lambda_group=lambda_group if use_penalty else 0.0,
            lambda_topo=lambda_topo,
            lambda_evidence=lambda_evidence,
        )
        self.energy = EnergyFunctional(
            self.dictionary, SmoothShapePrior(self.readout, shape_prior), weights
        )
        # Learned (positive) S-step size; the z-step uses the provable 1/L.
        self.log_s_step = nn.Parameter(torch.tensor(float(s_step_init)).log())

    # ---------------------------------------------------------------- utils
    @property
    def s_step(self) -> torch.Tensor:
        return self.log_s_step.exp()

    def thresholds(self, eta: torch.Tensor) -> Tuple[float, float]:
        if not self.sparse:
            return 0.0, 0.0
        return (
            float(eta) * self.energy.w.lambda_l1,
            float(eta) * self.energy.w.lambda_group,
        )

    def _z_step(self, z: torch.Tensor, s: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        v = z - eta * self.energy.grad_z_smooth(z, s)
        return self._project(v, eta)

    def _project(self, v: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        if not self.sparse:
            # Dense control: no prox and no projection at all -- not even a
            # zero-threshold shrinkage, which would still impose non-negativity
            # and thus smuggle in a structural asymmetry.
            return v
        if self.sparsity_mode == "topk":
            return project_topk_groups(v, self.topk_atoms, nonneg=self.nonneg_code,
                                       straight_through=self.training and self.straight_through)
        t_l1, t_grp = self.thresholds(eta)
        return prox_sparse_group(v, t_l1, t_grp, nonneg=self.nonneg_code)

    # -------------------------------------------------------------- forward
    def forward(
        self,
        x: torch.Tensor,
        n_steps: Optional[int] = None,
        intervene: Optional[Intervention] = None,
        adaptive: bool = False,
        plateau_eps: float = 1e-3,
        min_steps: int = 1,
        backtracking: bool = False,
        backtrack_shrink: float = 0.5,
        backtrack_max: int = 8,
        armijo_c: float = 1e-4,
        collect: bool = True,
        z0_override: Optional[torch.Tensor] = None,
    ) -> ReasoningOutput:
        K = self.n_steps if n_steps is None else int(n_steps)
        out_size = x.shape[-2:]

        g = self.encoder(x)                       # evidence, == S_0
        s = g
        if z0_override is not None:
            z = z0_override
        elif self.code_init is not None:
            z = self.code_init(s)
        else:
            z = torch.zeros(s.shape[0], self.dictionary.m, *s.shape[-2:],
                            device=s.device, dtype=s.dtype)

        eta = (1.0 / self.dictionary.lipschitz(refresh=self.training)).to(s.dtype)
        # z_0 must lie in the feasible set for the projected-gradient descent
        # argument to apply from the very first step.
        z = self._project(z, eta)

        trace = EnergyTrace()
        trace.append(self.energy.total(z, s, g))
        logits_per_step: List[torch.Tensor] = []
        codes: List[torch.Tensor] = []
        sketches: List[torch.Tensor] = [s]

        B = x.shape[0]
        active = torch.ones(B, dtype=torch.bool, device=x.device)
        steps_used = torch.zeros(B, device=x.device)

        create_graph = self.training and torch.is_grad_enabled()

        for t in range(K):
            z_new = self._z_step(z, s, eta)
            if intervene is not None:
                z_new = intervene(t, z_new)

            grad = self.energy.grad_s(z_new, s, g, create_graph=create_graph)
            if backtracking and not create_graph:
                obj = lambda ss: self.energy.smooth_in_s(z_new, ss, g)
                step_vec = torch.full((B,), float(self.s_step.detach()), device=s.device)
                s_new, _ = armijo_backtrack(
                    obj, s, grad, step_vec, backtrack_shrink, backtrack_max, armijo_c
                )
            else:
                s_new = s - self.s_step * grad

            # Freeze converged samples so adaptive depth is exact, not approximate.
            if adaptive:
                keep = active[:, None, None, None]
                z_new = torch.where(keep, z_new, z)
                s_new = torch.where(keep, s_new, s)

            z, s = z_new, s_new
            e = self.energy.total(z, s, g)
            prev = trace.values[-1].to(e.device)
            trace.append(e)
            steps_used = steps_used + active.float()

            if collect:
                codes.append(z)
                sketches.append(s)
                logits_per_step.append(self.readout(s, out_size))

            if adaptive and t + 1 >= min_steps:
                rel = (prev - e).abs() / (prev.abs() + 1e-8)
                active = active & (rel > plateau_eps)
                if not bool(active.any()):
                    break

        logits = logits_per_step[-1] if logits_per_step else self.readout(s, out_size)
        trace.steps_taken = steps_used.detach().cpu()
        return ReasoningOutput(
            logits=logits,
            logits_per_step=logits_per_step,
            codes=codes,
            sketches=sketches,
            energy=trace,
            steps_used=steps_used.detach(),
            evidence=g,
        )

    # ------------------------------------------------------------ interface
    @torch.no_grad()
    def predict(self, x: torch.Tensor, **kw) -> torch.Tensor:
        return torch.sigmoid(self.forward(x, **kw).logits)

    @torch.no_grad()
    def code_explained_variance(self, z: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        """1 - ||S - Dz||^2 / ||S||^2, per sample.

        The load-bearing question in one number.  If the code explains little of
        the sketch, the readout is reading something the code did not build, and
        no ablation of z can matter -- the reasoning state would be decorative
        *by construction*, which is precisely the failure mode this paper is
        about.  Reported for both the sparse model and the dense control.
        """
        resid = (s - self.dictionary.synthesize(z)).pow(2).flatten(1).sum(1)
        total = s.pow(2).flatten(1).sum(1).clamp_min(1e-8)
        return 1.0 - resid / total

    def on_optimizer_step(self) -> None:
        """Call after ``optimizer.step()``: renormalise atoms and refresh L."""
        self.dictionary.normalize_atoms()
        self.dictionary.lipschitz(refresh=True)

    def param_groups(self, lr: float, backbone_mult: float = 0.1,
                     weight_decay: float = 1e-4) -> List[Dict]:
        """Lower LR on the pretrained trunk; no weight decay on the dictionary
        (it is norm-constrained already, so decay would just fight the
        renormalisation) or on the step-size parameter."""
        trunk, no_decay, rest = [], [], []
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if name.startswith("encoder.trunk."):
                trunk.append(p)
            elif "dictionary.weight" in name or "log_s_step" in name:
                no_decay.append(p)
            else:
                rest.append(p)
        return [
            {"params": trunk, "lr": lr * backbone_mult, "weight_decay": weight_decay},
            {"params": rest, "lr": lr, "weight_decay": weight_decay},
            {"params": no_decay, "lr": lr, "weight_decay": 0.0},
        ]

In [ ]:
# ===== sparcseg/models/baselines.py ==========================================
"""Baselines, all sharing the SPARC-Seg backbone so the comparison is clean.

B1  SingleShot       -- same encoder + readout, one forward pass. Isolates
                        "does iterating help at all?".
B2  dense_unrolled   -- SPARCSeg(sparse=False). Same class, same K, same
                        parameter count, lambda_1 = lambda_group = 0. This is
                        the control the paper's central claim rests on.
B3  PTEALite         -- re-implementation of energy-based test-time refinement
                        (Progressive Test-Time Energy Adaptation, ICCV'25): a
                        learned dense plausibility energy, descended at test
                        time. Dense and non-interpretable by construction, which
                        is exactly the contrast we want.
B4  TextualBottleneck-- an on-theme replacement for "prompt a VLM to describe the
                        boundary". Instead of depending on an external VLM
                        (unreproducible, needs internet, confounded by a
                        different backbone), we force the *same* backbone to
                        route its prediction through a short sequence of
                        DISCRETE symbols -- a learned 'sentence' -- and render
                        the mask from those symbols alone. It is the cleanest
                        possible instantiation of the workshop's own claim that
                        a symbolic/textual medium cannot carry pixel-precise
                        spatial structure, and it is fully reproducible offline.
                        The literal VLM-prompting variant is available in
                        ``vlm_cot_available``/``run_vlm_cot`` when a local model
                        is mounted, and is reported as a secondary row.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

pass  # (inlined below)
pass  # (inlined below)


class SingleShot(nn.Module):
    """B1: no reasoning loop at all."""

    def __init__(self, sketch_dim: int = 64, backbone: str = "resnet34",
                 pretrained: bool = True, in_channels: int = 3,
                 n_classes: int = 1) -> None:
        super().__init__()
        self.encoder = Encoder(backbone, pretrained, sketch_dim, in_channels)
        self.readout = ReadoutHead(sketch_dim, n_classes=n_classes, scale=4)

    def forward(self, x: torch.Tensor, **_: object) -> ReasoningOutput:
        s = self.encoder(x)
        logits = self.readout(s, x.shape[-2:])
        return ReasoningOutput(logits=logits, logits_per_step=[logits],
                               sketches=[s], evidence=s,
                               steps_used=torch.zeros(x.shape[0], device=x.device))

    def param_groups(self, lr: float, backbone_mult: float = 0.1,
                     weight_decay: float = 1e-4) -> List[Dict]:
        trunk = [p for n, p in self.named_parameters() if n.startswith("encoder.trunk.")]
        rest = [p for n, p in self.named_parameters() if not n.startswith("encoder.trunk.")]
        return [
            {"params": trunk, "lr": lr * backbone_mult, "weight_decay": weight_decay},
            {"params": rest, "lr": lr, "weight_decay": weight_decay},
        ]

    def on_optimizer_step(self) -> None:
        return None


class PlausibilityEnergy(nn.Module):
    """Dense, learned scalar energy over (image evidence, soft mask)."""

    def __init__(self, sketch_dim: int = 64, width: int = 32) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(sketch_dim + 1, width, 3, padding=1), nn.GroupNorm(4, width), nn.SiLU(),
            nn.Conv2d(width, width, 3, padding=1, stride=2), nn.GroupNorm(4, width), nn.SiLU(),
            nn.Conv2d(width, width, 3, padding=1, stride=2), nn.GroupNorm(4, width), nn.SiLU(),
        )
        self.head = nn.Linear(width, 1)

    def forward(self, s: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
        h = self.net(torch.cat([s, p], dim=1))
        return self.head(h.mean(dim=(2, 3))).squeeze(-1)   # (B,)


class PTEALite(nn.Module):
    """B3: predict, then descend a learned plausibility energy at test time."""

    def __init__(self, sketch_dim: int = 64, backbone: str = "resnet34",
                 pretrained: bool = True, in_channels: int = 3,
                 n_classes: int = 1, n_steps: int = 4, step_size: float = 1.0) -> None:
        super().__init__()
        self.encoder = Encoder(backbone, pretrained, sketch_dim, in_channels)
        self.readout = ReadoutHead(sketch_dim, n_classes=n_classes, scale=4)
        self.energy = PlausibilityEnergy(sketch_dim)
        self.n_steps = n_steps
        self.step_size = step_size

    def forward(self, x: torch.Tensor, n_steps: Optional[int] = None,
                adapt: Optional[bool] = None, **_: object) -> ReasoningOutput:
        K = self.n_steps if n_steps is None else int(n_steps)
        do_adapt = (not self.training) if adapt is None else adapt
        s = self.encoder(x)
        logits_lr = self.readout.logits_at_sketch_res(s)
        per_step: List[torch.Tensor] = []

        if do_adapt and K > 0:
            cur = logits_lr.detach().clone()
            for _ in range(K):
                with torch.enable_grad():
                    cur = cur.detach().requires_grad_(True)
                    e = self.energy(s.detach(), torch.sigmoid(cur)).sum()
                    (g,) = torch.autograd.grad(e, cur)
                cur = (cur - self.step_size * g).detach()
                per_step.append(F.interpolate(cur, size=x.shape[-2:],
                                              mode="bilinear", align_corners=False))
            logits = per_step[-1]
        else:
            logits = F.interpolate(logits_lr, size=x.shape[-2:],
                                   mode="bilinear", align_corners=False)
            per_step = [logits]

        return ReasoningOutput(logits=logits, logits_per_step=per_step,
                               sketches=[s], evidence=s,
                               steps_used=torch.full((x.shape[0],), float(K),
                                                     device=x.device))

    def energy_training_loss(self, x: torch.Tensor, gt: torch.Tensor,
                             margin: float = 1.0) -> torch.Tensor:
        """Margin loss: the ground-truth mask must score lower than a corrupted
        one. Without this the energy is free to be constant and the test-time
        descent becomes a no-op -- which would make B3 a strawman."""
        with torch.no_grad():
            s = self.encoder(x)
            gt_lr = F.interpolate(gt, size=s.shape[-2:], mode="area")
            noise = torch.rand_like(gt_lr)
            corrupt = torch.where(noise < 0.15, 1.0 - gt_lr, gt_lr)
            shift = torch.roll(gt_lr, shifts=(3, 3), dims=(2, 3))
            corrupt = torch.where(torch.rand_like(gt_lr) < 0.5, corrupt, shift)
        e_pos = self.energy(s, gt_lr)
        e_neg = self.energy(s, corrupt)
        return F.relu(margin + e_pos - e_neg).mean()

    def param_groups(self, lr: float, backbone_mult: float = 0.1,
                     weight_decay: float = 1e-4) -> List[Dict]:
        trunk = [p for n, p in self.named_parameters() if n.startswith("encoder.trunk.")]
        rest = [p for n, p in self.named_parameters() if not n.startswith("encoder.trunk.")]
        return [
            {"params": trunk, "lr": lr * backbone_mult, "weight_decay": weight_decay},
            {"params": rest, "lr": lr, "weight_decay": weight_decay},
        ]

    def on_optimizer_step(self) -> None:
        return None


class TextualBottleneck(nn.Module):
    """B4: reasoning forced through a discrete, language-like bottleneck.

    The encoder's spatial evidence is pooled to a global vector, quantised into
    ``n_tokens`` symbols drawn from a ``vocab_size`` codebook (straight-through
    Gumbel-softmax), and the mask is rendered from the token embeddings alone --
    no spatial skip connection survives.  The bottleneck therefore carries
    ``n_tokens * log2(vocab_size)`` bits, the same order as a short sentence.

    This is the controlled version of "narrate the boundary in words, then draw
    it": same backbone, same training budget, same loss -- only the medium of
    the intermediate state changes.  Any boundary-F gap is attributable to the
    medium rather than to a different model family.
    """

    def __init__(self, sketch_dim: int = 64, backbone: str = "resnet34",
                 pretrained: bool = True, in_channels: int = 3, n_classes: int = 1,
                 n_tokens: int = 32, vocab_size: int = 256, embed_dim: int = 64,
                 out_stride: int = 4, tau_start: float = 2.0, tau_end: float = 0.5,
                 anneal_steps: int = 2000) -> None:
        super().__init__()
        self.encoder = Encoder(backbone, pretrained, sketch_dim, in_channels)
        self.n_tokens, self.vocab_size = n_tokens, vocab_size
        self.tau_start, self.tau_end, self.anneal_steps = tau_start, tau_end, anneal_steps
        self.register_buffer("_train_steps", torch.zeros((), dtype=torch.long))
        self.out_stride = out_stride
        self.to_logits = nn.Sequential(
            nn.Linear(sketch_dim, 256), nn.SiLU(), nn.Linear(256, n_tokens * vocab_size)
        )
        self.codebook = nn.Embedding(vocab_size, embed_dim)
        self.render = nn.Sequential(
            nn.Linear(n_tokens * embed_dim, 512), nn.SiLU(),
            nn.Linear(512, 8 * 8 * 32), nn.SiLU(),
        )
        # Note: the decoding path here is deliberately given MORE capacity than
        # the other baselines' readout heads. If a symbolic bottleneck still
        # loses on boundary metrics with a decoder this generous, the medium --
        # not the decoder -- is what is costing the boundary precision.
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(32, 32, 4, 2, 1), nn.GroupNorm(4, 32), nn.SiLU(),
            nn.ConvTranspose2d(32, 32, 4, 2, 1), nn.GroupNorm(4, 32), nn.SiLU(),
            nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.GroupNorm(4, 16), nn.SiLU(),
            nn.Conv2d(16, n_classes, 3, padding=1),
        )

    def bits(self) -> float:
        """Capacity of the symbolic bottleneck, quoted in the paper alongside
        the result so the comparison is stated in information terms."""
        import math
        return self.n_tokens * math.log2(self.vocab_size)

    def forward(self, x: torch.Tensor, **_: object) -> ReasoningOutput:
        B = x.shape[0]
        s = self.encoder(x)
        pooled = s.mean(dim=(2, 3))
        tok_logits = self.to_logits(pooled).view(B, self.n_tokens, self.vocab_size)
        if self.training:
            # Anneal the Gumbel temperature: a fixed tau=1 discretises too early
            # and the baseline never learns, which would make it a strawman
            # rather than a fair test of the symbolic medium.
            frac = min(1.0, float(self._train_steps) / max(self.anneal_steps, 1))
            tau = self.tau_start + frac * (self.tau_end - self.tau_start)
            onehot = F.gumbel_softmax(tok_logits, tau=tau, hard=True, dim=-1)
            self._train_steps += 1
        else:
            idx = tok_logits.argmax(dim=-1)
            onehot = F.one_hot(idx, self.vocab_size).to(tok_logits.dtype)
        emb = onehot @ self.codebook.weight              # (B, n_tokens, embed)
        h = self.render(emb.flatten(1)).view(B, 32, 8, 8)
        logits = self.upsample(h)
        logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear",
                               align_corners=False)
        return ReasoningOutput(logits=logits, logits_per_step=[logits],
                               sketches=[s], evidence=s,
                               steps_used=torch.zeros(B, device=x.device))

    def param_groups(self, lr: float, backbone_mult: float = 0.1,
                     weight_decay: float = 1e-4) -> List[Dict]:
        trunk = [p for n, p in self.named_parameters() if n.startswith("encoder.trunk.")]
        rest = [p for n, p in self.named_parameters() if not n.startswith("encoder.trunk.")]
        return [
            {"params": trunk, "lr": lr * backbone_mult, "weight_decay": weight_decay},
            {"params": rest, "lr": lr, "weight_decay": weight_decay},
        ]

    def on_optimizer_step(self) -> None:
        return None


# --------------------------------------------------------------------------
# Factory
# --------------------------------------------------------------------------
def build_model(method: str, cfg, in_channels: int = 3, n_classes: int = 1) -> nn.Module:
    common = dict(sketch_dim=cfg.sketch_dim, backbone=cfg.backbone,
                  pretrained=cfg.pretrained, in_channels=in_channels,
                  n_classes=n_classes)
    if method == "singleshot":
        return SingleShot(**common)
    if method == "ptea_lite":
        return PTEALite(n_steps=cfg.n_steps, **common)
    if method == "textual_bottleneck":
        return TextualBottleneck(**common)
    if method in {"sparcseg", "dense_unrolled"}:
        return SPARCSeg(
            dict_size=cfg.dict_size,
            n_steps=cfg.n_steps,
            lambda_l1=cfg.lambda_l1,
            lambda_group=cfg.lambda_group,
            lambda_topo=cfg.lambda_topo,
            lambda_evidence=cfg.lambda_evidence,
            s_step_init=cfg.s_step_init,
            nonneg_code=cfg.nonneg_code,
            sparse=(method == "sparcseg"),
            sparsity_mode=getattr(cfg, "sparsity_mode", "topk"),
            topk_atoms=getattr(cfg, "topk_atoms", 8),
            straight_through=getattr(cfg, "straight_through", True),
            **common,
        )
    raise ValueError(f"unknown method: {method!r}")


def is_reasoning_model(model: nn.Module) -> bool:
    return isinstance(model, SPARCSeg)


# --------------------------------------------------------------------------
# Optional: the literal VLM chain-of-thought baseline
# --------------------------------------------------------------------------
def vlm_cot_available() -> bool:
    """True only if transformers and a locally mounted VLM are both present.

    Deliberately never downloads: a baseline that silently needs internet is a
    baseline your co-authors cannot reproduce.
    """
    try:
        import transformers  # noqa: F401
    except Exception:
        return False
    from pathlib import Path
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    for p in base.rglob("config.json"):
        try:
            import json
            cfg = json.loads(p.read_text())
        except Exception:
            continue
        if "vision_config" in cfg or "vision_tower" in str(cfg).lower():
            return True
    return False

In [ ]:
# ===== sparcseg/losses.py ====================================================
"""Training objectives.

Beyond the usual BCE + soft-Dice, two auxiliaries matter for this paper:

``atom_usage_balance``  Sparse dictionaries collapse. Left alone, training
    happily converges to a state where 4 of 192 atoms carry everything and the
    rest are dead -- at which point "sparse and interpretable" is true but
    vacuous, and the sufficiency score is trivially 1.0. A load-balancing term
    (same idea as the auxiliary loss used for mixture-of-experts routing) keeps
    the dictionary populated. Its weight is reported, and an ablation with the
    term removed is included, because a reviewer will ask whether the
    interpretability is an artefact of this term.

``step_monotonicity``  The energy is guaranteed to decrease, but nothing
    guarantees the *mask* improves. Penalising per-step Dice regressions makes
    the two coincide in practice and gives the paper its honest answer to
    "so what if E decreases?".
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor,
                   eps: float = 1.0) -> torch.Tensor:
    p = torch.sigmoid(logits)
    num = 2.0 * (p * target).flatten(1).sum(1) + eps
    den = p.flatten(1).sum(1) + target.flatten(1).sum(1) + eps
    return (1.0 - num / den).mean()


def bce_loss(logits: torch.Tensor, target: torch.Tensor,
             pos_weight: Optional[torch.Tensor] = None) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)


def seg_loss(logits: torch.Tensor, target: torch.Tensor, alpha: float = 0.5,
             pos_weight: Optional[torch.Tensor] = None) -> torch.Tensor:
    return alpha * bce_loss(logits, target, pos_weight) + (1 - alpha) * soft_dice_loss(logits, target)


def deep_supervision_loss(
    logits_per_step: Sequence[torch.Tensor],
    target: torch.Tensor,
    alpha: float = 0.5,
    decay: float = 0.5,
    pos_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Weight later steps more: weight_t = decay ** (K - 1 - t), normalised.

    Supervising every step is what makes the intermediate sketches actually
    decodable -- without it, S_1..S_{K-1} are free to be arbitrary and the
    step-wise interpretability story has nothing to stand on.
    """
    if not logits_per_step:
        raise ValueError("deep_supervision_loss called with no steps")
    K = len(logits_per_step)
    weights = [decay ** (K - 1 - t) for t in range(K)]
    total_w = sum(weights)
    loss = logits_per_step[0].new_zeros(())
    for w, lg in zip(weights, logits_per_step):
        loss = loss + (w / total_w) * seg_loss(lg, target, alpha, pos_weight)
    return loss


def atom_usage_balance(z: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Encourage activation mass to spread across atoms (lower = better spread).

    Uses the coefficient of variation of per-atom mass, which is scale-free and
    therefore does not fight the L1 term for control of the overall magnitude.
    """
    mass = z.detach().abs().flatten(2).sum(2).mean(0) if z.requires_grad is False else \
        z.abs().flatten(2).sum(2).mean(0)                      # (m,)
    mean = mass.mean() + eps
    return mass.std(unbiased=False) / mean


def step_monotonicity_penalty(logits_per_step: Sequence[torch.Tensor],
                              target: torch.Tensor) -> torch.Tensor:
    """Penalise any step whose soft-Dice is worse than the previous step's."""
    if len(logits_per_step) < 2:
        return logits_per_step[0].new_zeros(()) if logits_per_step else torch.zeros(())
    pen = logits_per_step[0].new_zeros(())
    prev = soft_dice_loss(logits_per_step[0], target)
    for lg in logits_per_step[1:]:
        cur = soft_dice_loss(lg, target)
        pen = pen + F.relu(cur - prev)
        prev = cur
    return pen / (len(logits_per_step) - 1)


class SPARCSegLoss(nn.Module):
    def __init__(self, alpha: float = 0.5, deep_decay: float = 0.5,
                 w_usage: float = 0.01, w_monotone: float = 0.05,
                 deep_supervision: bool = True) -> None:
        super().__init__()
        self.alpha = alpha
        self.deep_decay = deep_decay
        self.w_usage = w_usage
        self.w_monotone = w_monotone
        self.deep_supervision = deep_supervision

    def forward(self, out, target: torch.Tensor,
                pos_weight: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        steps = out.logits_per_step or [out.logits]
        if self.deep_supervision and len(steps) > 1:
            main = deep_supervision_loss(steps, target, self.alpha, self.deep_decay, pos_weight)
        else:
            main = seg_loss(out.logits, target, self.alpha, pos_weight)

        parts: Dict[str, torch.Tensor] = {"seg": main}
        total = main

        if out.codes and self.w_usage > 0:
            usage = atom_usage_balance(out.codes[-1])
            parts["usage"] = usage
            total = total + self.w_usage * usage

        if len(steps) > 1 and self.w_monotone > 0:
            mono = step_monotonicity_penalty(steps, target)
            parts["monotone"] = mono
            total = total + self.w_monotone * mono

        parts["total"] = total
        return parts


def positive_weight_from_loader(loader, max_batches: int = 20,
                                device: Optional[torch.device] = None) -> torch.Tensor:
    """pos_weight = #neg / #pos, estimated from a few batches.

    Lesions occupy a few percent of the frame in BUSI and BRISC; without this
    BCE is dominated by background and early training collapses to all-zeros --
    from which the reasoning loop has nothing to revise.
    """
    pos = neg = 0.0
    for i, batch in enumerate(loader):
        y = batch["mask"]
        pos += float(y.sum())
        neg += float(y.numel() - y.sum())
        if i + 1 >= max_batches:
            break
    ratio = neg / max(pos, 1.0)
    return torch.tensor(min(max(ratio, 1.0), 20.0), device=device)

In [ ]:
# ===== sparcseg/concepts.py ==================================================
"""Objective concept naming for dictionary atoms.

The proposal illustrates interpretability with hand-written labels ("atom #14
'sharp boundary', atom #31 'hypoechoic texture'").  Hand labelling is the single
easiest thing for a reviewer to dismiss, and rightly: it is unfalsifiable and
unreproducible across the three dataset owners.

Here an atom is named by *measurement*.  For each atom we correlate its spatial
activation map against a fixed battery of measurable per-pixel image statistics,
and the name is whichever statistic it tracks most strongly, reported together
with the correlation, a permutation-based significance value, and a stability
score across folds.  An atom whose best correlation is weak is labelled
``unnamed`` rather than given a flattering story -- which is itself a result
worth reporting honestly.

The battery is deliberately physics-diverse so that the same code names atoms
sensibly on ultrasound, dermoscopy and MRI without per-dataset tweaking:
gradient/edge, texture at two scales, local contrast, absolute and relative
intensity, colour saturation, and distance to the annotated boundary.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
import torch.nn.functional as F

pass  # (inlined below)
pass  # (inlined below)


# Atoms are NAMED using image-intrinsic statistics only. The two
# ground-truth-referenced descriptors below are computed as well, but are
# excluded from naming: an atom called "boundary_distance" would be describing
# the annotation rather than the image, and in an early run those two GT-derived
# descriptors captured 35 of 40 named atoms purely because they are the
# strongest available correlates of anything the network learned. They are
# reported separately as a LOCALISATION diagnostic, which is what they honestly
# measure.
INTRINSIC_DESCRIPTORS = [
    "edge_gradient",        # |grad I| -- sharp boundary evidence
    "texture_fine",         # local std, 3x3 -- speckle / fine texture
    "texture_coarse",       # local std, 11x11 -- regional heterogeneity
    "local_contrast",       # centre-surround difference
    "intensity_bright",     # absolute brightness (hyperechoic / enhancing)
    "intensity_dark",       # darkness (hypoechoic / necrotic)
    "saturation",           # colour purity -- matters on dermoscopy, flat on MRI
]

GT_REFERENCED_DESCRIPTORS = [
    "boundary_distance",    # proximity to the annotated lesion boundary
    "interior_distance",    # depth inside the lesion
]

DESCRIPTOR_NAMES = INTRINSIC_DESCRIPTORS + GT_REFERENCED_DESCRIPTORS


def denormalize(x: torch.Tensor) -> np.ndarray:
    """(3,H,W) normalized tensor -> HxWx3 uint8 RGB."""
    img = x.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img * 255.0, 0, 255).astype(np.uint8)


def pixel_descriptors(rgb: np.ndarray, gt: Optional[np.ndarray] = None) -> Dict[str, np.ndarray]:
    """Per-pixel measurable statistics, each returned as float32 HxW."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx ** 2 + gy ** 2)

    def local_std(img: np.ndarray, k: int) -> np.ndarray:
        mu = cv2.blur(img, (k, k))
        mu2 = cv2.blur(img * img, (k, k))
        return np.sqrt(np.maximum(mu2 - mu * mu, 0.0))

    blur_small = cv2.GaussianBlur(gray, (5, 5), 0)
    blur_large = cv2.GaussianBlur(gray, (21, 21), 0)

    out: Dict[str, np.ndarray] = {
        "edge_gradient": grad,
        "texture_fine": local_std(gray, 3),
        "texture_coarse": local_std(gray, 11),
        "local_contrast": np.abs(blur_small - blur_large),
        "intensity_bright": gray,
        "intensity_dark": 1.0 - gray,
        "saturation": hsv[..., 1] / 255.0,
    }

    if gt is not None and gt.any():
        g = gt.astype(np.uint8)
        d_in = cv2.distanceTransform(g, cv2.DIST_L2, 3)
        d_out = cv2.distanceTransform(1 - g, cv2.DIST_L2, 3)
        signed = d_in - d_out
        out["boundary_distance"] = np.exp(-np.abs(signed) / 8.0).astype(np.float32)
        out["interior_distance"] = (d_in / (d_in.max() + 1e-6)).astype(np.float32)
    else:
        z = np.zeros_like(gray)
        out["boundary_distance"] = z
        out["interior_distance"] = z

    return {k: v.astype(np.float32) for k, v in out.items()}


@dataclass
class AtomProfile:
    atom_id: int
    name: str
    rho: float
    p_value: float
    all_rho: Dict[str, float]
    activation_mass: float
    n_images_active: int

    def as_row(self) -> Dict[str, object]:
        return {"atom": self.atom_id, "name": self.name, "rho": self.rho,
                "p": self.p_value, "mass": self.activation_mass,
                "n_active": self.n_images_active}


@torch.no_grad()
def profile_atoms(
    model,
    loader,
    device: torch.device,
    max_images: int = 200,
    max_pixels_per_image: int = 2048,
    min_abs_rho: float = 0.15,
    seed: int = 0,
    step: int = -1,
) -> Dict[str, object]:
    """Correlate every atom's activation against the descriptor battery.

    Pixels are subsampled per image (``max_pixels_per_image``) because adjacent
    pixels are heavily correlated: using all of them would inflate the effective
    sample size and make every p-value look spectacular for the wrong reason.
    """
    model.eval()
    m = model.dictionary.m
    rng = np.random.default_rng(seed)

    acts: Dict[int, List[np.ndarray]] = {j: [] for j in range(m)}
    descs: Dict[int, Dict[str, List[np.ndarray]]] = {
        j: {k: [] for k in DESCRIPTOR_NAMES} for j in range(m)
    }
    mass = np.zeros(m, dtype=np.float64)
    n_active = np.zeros(m, dtype=np.int64)
    seen = 0

    for batch in loader:
        if seen >= max_images:
            break
        x = batch["image"].to(device)
        gts = (batch["mask"].squeeze(1).numpy() > 0.5)
        out = model(x, n_steps=model.n_steps)
        z = out.codes[step]                                   # (B, m, h, w)
        H, W = gts.shape[-2:]
        z_up = F.interpolate(z, size=(H, W), mode="bilinear", align_corners=False)

        for b in range(x.shape[0]):
            rgb = denormalize(x[b])
            d = pixel_descriptors(rgb, gts[b])
            flat_d = {k: v.ravel() for k, v in d.items()}
            n_pix = H * W
            idx = rng.choice(n_pix, size=min(max_pixels_per_image, n_pix), replace=False)

            zb = z_up[b].cpu().numpy()
            for j in range(m):
                a = zb[j].ravel()[idx]
                total = float(np.abs(zb[j]).sum())
                mass[j] += total
                if total <= 1e-8 or np.allclose(a, a[0]):
                    continue
                n_active[j] += 1
                acts[j].append(a)
                for k in DESCRIPTOR_NAMES:
                    descs[j][k].append(flat_d[k][idx])
        seen += x.shape[0]

    profiles: List[AtomProfile] = []
    for j in range(m):
        if not acts[j]:
            profiles.append(AtomProfile(j, "dead", float("nan"), float("nan"),
                                        {}, float(mass[j]), 0))
            continue
        a = np.concatenate(acts[j])
        rhos: Dict[str, float] = {}
        ps: Dict[str, float] = {}
        for k in DESCRIPTOR_NAMES:
            v = np.concatenate(descs[j][k])
            r = spearman(a, v)
            rhos[k] = r["rho"]
            ps[k] = r["p"]
        best = max(INTRINSIC_DESCRIPTORS,
                   key=lambda k: abs(rhos[k]) if np.isfinite(rhos[k]) else -1)
        best_rho = rhos[best]
        name = best if (np.isfinite(best_rho) and abs(best_rho) >= min_abs_rho) else "unnamed"
        if name != "unnamed" and best_rho < 0:
            name = f"anti_{best}"
        profiles.append(AtomProfile(j, name, float(best_rho), float(ps[best]),
                                    rhos, float(mass[j]), int(n_active[j])))

    named = [p for p in profiles if p.name not in {"dead", "unnamed"}]
    loc = [max((abs(p.all_rho.get(k, np.nan)) for k in GT_REFERENCED_DESCRIPTORS),
               default=np.nan) for p in profiles if p.all_rho]
    loc = [v for v in loc if np.isfinite(v)]
    vocab: Dict[str, int] = {}
    for p in named:
        vocab[p.name] = vocab.get(p.name, 0) + 1

    return {
        "profiles": [p.as_row() for p in profiles],
        "full": {p.atom_id: p.all_rho for p in profiles},
        "summary": {
            "n_atoms": float(len(profiles)),
            "n_dead": float(sum(1 for p in profiles if p.name == "dead")),
            "n_named": float(len(named)),
            "named_fraction": float(len(named) / max(len(profiles), 1)),
            "mean_abs_rho_named": float(np.mean([abs(p.rho) for p in named])) if named else float("nan"),
            "mean_localisation_rho": float(np.mean(loc)) if loc else float("nan"),
            "vocabulary": vocab,
        },
    }


def naming_stability(profiles_a: Sequence[Dict], profiles_b: Sequence[Dict]) -> float:
    """Fraction of atoms that receive the same name in two independent runs.

    Atom indices are not comparable across runs with different seeds, so this is
    only meaningful between folds of the *same* initialisation -- which is how it
    is used in ``experiment.py``. Reported so the interpretability claim comes
    with a reproducibility number rather than a single cherry-picked figure.
    """
    a = {p["atom"]: p["name"] for p in profiles_a}
    b = {p["atom"]: p["name"] for p in profiles_b}
    shared = [k for k in a if k in b and a[k] not in {"dead", "unnamed"}]
    if not shared:
        return float("nan")
    return float(np.mean([a[k] == b[k] for k in shared]))


def top_atoms_table(result: Dict[str, object], k: int = 15) -> List[Dict[str, object]]:
    rows = [r for r in result["profiles"] if r["name"] != "dead"]
    rows.sort(key=lambda r: -r["mass"])
    return rows[:k]

---
## Part 3 - Causal faithfulness: the paper's centrepiece

The original protocol - ablate one atom in the sparse model, one channel in the
dense model, show the sparse drop is bigger - **is confounded**, and it is the
first thing a competent reviewer will say. With $k$ active atoms, zeroing one
removes $\sim1/k$ of the state's energy; with $m$ dense channels it removes
$\sim1/m$. Since $k\ll m$, the sparse model shows a larger drop *whatever its
internal structure*. That measures the sparsity level, not whether the state is
load-bearing.

Everything here is built to remove that confound.

| Test | Question it answers | Why it is here |
|---|---|---|
| **Norm-matched ablation** | at matched removed *energy* $\rho$, does it matter *which* units go? | removes the $1/k$ vs $1/m$ confound outright |
| **Support-restricted null** | same $\rho$, random units **from the support** | a uniform random null over all $m$ units degenerates for a sparse code: it spends its early picks on zero-energy units and converges onto the same set, driving CSI to ~0 by construction |
| **CSI** | area between ordered curve and null, normalised | scale-free "is the state organised?" - a decorative dense state scores ~0 even when its naive drop is large |
| **Sufficiency** | keep only the top units carrying $\rho$ | the complement of necessity |
| **Transplantation** | patch donor A's code into recipient B | *direction*: B must move **toward A's content**, not merely away from B's. Noise also destroys a mask; only a content-bearing state transfers content |
| **Steering** | scale one atom by $\alpha$ | turns a name into a falsifiable prediction about a measurable output property |
| **Spatial alignment** | does removing an atom change the mask *where that atom was active*? | faithfulness that is also localised |
| **Bottleneck diagnostics** | how much of $S$ does the code explain? | the structural precondition for all of the above. Without it, a null result is unreadable: was the state decorative, or was the intervention simply too weak? |

Naive top-1 necessity is still computed and printed, **labelled as confounded**,
so the numbers stay comparable with how the rest of the literature reports it.


In [ ]:
# ===== sparcseg/causal.py ====================================================
"""The causal-faithfulness protocol.

Why this file is not just "zero an atom and measure Dice"
---------------------------------------------------------
The original proposal's headline comparison -- ablate one active atom in the
sparse model, ablate one channel in the dense model, show the sparse drop is
bigger -- is confounded, and a competent reviewer will say so in one line.  With
k active atoms, zeroing one removes ~1/k of the state's energy; with m dense
channels it removes ~1/m.  Since k << m, the sparse model must show a larger
drop *whatever its internal structure*.  The result would measure the sparsity
level, not whether the state is load-bearing.

Everything here is built around removing that confound:

1. NORM-MATCHED ABLATION.  Both models are ablated at a matched *fraction of
   removed reconstruction energy* rho, not a matched number of units.  The
   achieved removed energy is recorded for both so the match can be reported.

2. RANDOM-DIRECTION NULL.  At each rho we also ablate a randomly chosen unit set
   carrying the same energy.  The reported quantity is the *gap* between the
   importance-ordered curve and this null.

3. CAUSAL STRUCTURE INDEX (CSI).  The area between those two curves, normalised
   by the baseline Dice.  CSI is invariant to how much energy a unit happens to
   carry; it answers "is *which* unit you remove what matters?" -- i.e. is the
   state organised, or is it an undifferentiated activation blob.  A decorative
   dense state scores ~0 even when its naive ablation drop is large.

4. STATE TRANSPLANTATION (interchange intervention).  Donor A's code is patched
   into recipient B's run.  If the state is load-bearing, B's output moves
   *toward A's content*, not merely away from B's.  Directionality is what
   separates causation from damage: noise also destroys a mask.

5. STEERING.  Scaling one atom's coefficient should move a *named, measurable*
   property of the output monotonically. This turns interpretability from a
   picture into a testable prediction.

6. SPATIAL ALIGNMENT.  Knocking out an atom should change the mask where that
   atom was active. Faithfulness that is also localised.

Naive top-1 necessity is still computed and reported -- labelled as confounded --
so the numbers can be compared against how the rest of the literature reports it.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

pass  # (inlined below)
pass  # (inlined below)


# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------
@dataclass
class CausalConfig:
    fractions: Tuple[float, ...] = (0.05, 0.1, 0.2, 0.3, 0.5, 0.7)
    n_random: int = 8
    max_images: int = 300
    batch_size: int = 8
    seed: int = 0
    # True: the unit is knocked out at EVERY step (process necessity) -- the
    # standard knockout in causal mediation, and the right primary measure.
    # Ablating only at the final step was tried first and is structurally weak:
    # a single S-step after the ablation barely moves the sketch, so necessity
    # came out at ~0 for *both* models and the comparison had no dynamic range.
    # Last-step ablation is still available (persistent=False) and reported as a
    # secondary "readout necessity" number.
    persistent: bool = True
    threshold: float = 0.5
    steering_alphas: Tuple[float, ...] = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0)
    n_transplant_pairs: int = 200
    top_atoms_for_steering: int = 8


# --------------------------------------------------------------------------
# Low-level helpers
# --------------------------------------------------------------------------
def unit_energy(z: torch.Tensor) -> torch.Tensor:
    """Per-unit squared contribution proxy ||z_j||^2, shape (B, m).

    Exact when dictionary atoms are orthonormal; atoms are unit-norm by
    construction and ``ConceptDictionary.coherence()`` reports how far from
    orthogonal they actually are, so the approximation is auditable rather than
    assumed.  Selection uses this proxy; the *achieved* removed energy is
    measured exactly afterwards.
    """
    return z.flatten(2).pow(2).sum(2)


def exact_removed_energy(model: SPARCSeg, z: torch.Tensor,
                         keep_mask: torch.Tensor) -> torch.Tensor:
    """|| D z - D (z * keep) ||^2 per sample -- the true removed energy."""
    with torch.no_grad():
        full = model.dictionary.synthesize(z)
        kept = model.dictionary.synthesize(z * keep_mask)
        return (full - kept).pow(2).flatten(1).sum(1)


def select_units_by_energy(
    energies: torch.Tensor, fraction: float, order: str = "top", seed: int = 0
) -> torch.Tensor:
    """Choose, per sample, the smallest unit set whose energy share >= fraction.

    ``order='top'``     -> importance-ordered, largest contribution first
    ``order='random'``  -> the matched null: a random order *restricted to the
                           support* (units that actually carry energy)
    ``order='bottom'``  -> smallest contribution first; the hardest contrast,
                           since it removes the same energy from many small units

    Returns a float keep-mask of shape (B, m, 1, 1): 1 = keep, 0 = ablate.

    Why the null is support-restricted
    ----------------------------------
    A uniformly random permutation over all m units is the obvious null and it
    is WRONG for a hard-sparse code.  With 8 of 48 units carrying all the energy,
    a random permutation spends its early picks on zero-energy units, which cost
    nothing and are skipped by the cumulative-energy rule -- so the "random" set
    converges to almost exactly the importance-ordered set and CSI collapses to
    ~0 by construction.  (Measured: CSI -0.0016 for a model whose ablations were
    otherwise behaving perfectly.)  Restricting the null to the support asks the
    question that actually matters: given that we remove this much energy *from
    units that carry energy*, does it matter WHICH ones?
    """
    B, m = energies.shape
    total = energies.sum(dim=1, keepdim=True).clamp_min(1e-12)
    share = energies / total

    if order == "top":
        idx = torch.argsort(share, dim=1, descending=True)
    elif order == "bottom":
        # Ascending among the support; zero-energy units last so they never
        # pad the set.
        key = torch.where(share > 0, share, torch.full_like(share, 2.0))
        idx = torch.argsort(key, dim=1, descending=False)
    elif order == "random":
        g = torch.Generator(device="cpu").manual_seed(seed)
        r = torch.rand(B, m, generator=g).to(energies.device)
        key = torch.where(share > 0, r, r + 2.0)
        idx = torch.argsort(key, dim=1, descending=False)
    else:
        raise ValueError(order)

    sorted_share = torch.gather(share, 1, idx)
    cumulative = sorted_share.cumsum(dim=1)
    # Include the unit that crosses the threshold, so achieved energy >= fraction.
    take = (cumulative - sorted_share) < fraction
    ablate = torch.zeros_like(share, dtype=torch.bool)
    ablate.scatter_(1, idx, take)
    keep = (~ablate).to(energies.dtype)
    return keep[:, :, None, None]


def make_ablation_hook(keep: torch.Tensor, step: Optional[int],
                       n_steps: int) -> Callable[[int, torch.Tensor], torch.Tensor]:
    """Intervention hook. ``step=None`` knocks the units out at every step
    (process necessity); an integer ablates only at that step (readout
    necessity, no opportunity for the loop to repair the damage)."""
    target = (n_steps - 1) if step is None else step

    def hook(t: int, z: torch.Tensor) -> torch.Tensor:
        if step is None:
            return z * keep
        return z * keep if t == target else z

    return hook


def _hook_persistent(keep: torch.Tensor) -> Callable[[int, torch.Tensor], torch.Tensor]:
    def hook(t: int, z: torch.Tensor) -> torch.Tensor:
        return z * keep
    return hook


def binarize(logits: torch.Tensor, threshold: float = 0.5) -> np.ndarray:
    return (torch.sigmoid(logits) > threshold).squeeze(1).cpu().numpy()


def batch_dice(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    return np.array([dice_score(p, g) for p, g in zip(pred, gt)], dtype=np.float64)


# --------------------------------------------------------------------------
# Necessity / sufficiency curves
# --------------------------------------------------------------------------
@torch.no_grad()
def ablation_curves(
    model: SPARCSeg,
    loader,
    device: torch.device,
    cfg: CausalConfig,
    n_steps: Optional[int] = None,
) -> Dict[str, object]:
    """Norm-matched necessity and sufficiency curves plus the random null."""
    model.eval()
    K = n_steps or model.n_steps
    fr = list(cfg.fractions)

    per_image: List[Dict[str, float]] = []
    seen = 0

    for batch in loader:
        if seen >= cfg.max_images:
            break
        x = batch["image"].to(device)
        y = batch["mask"].squeeze(1).cpu().numpy() > 0.5

        base = model(x, n_steps=K)
        base_pred = binarize(base.logits, cfg.threshold)
        base_dice = batch_dice(base_pred, y)
        z_final = base.codes[-1]
        energies = unit_energy(z_final)
        active = (energies > 1e-10).float().sum(1).cpu().numpy()

        rows: List[Dict[str, float]] = [
            {"base_dice": float(d), "n_active_units": float(a)}
            for d, a in zip(base_dice, active)
        ]

        for f in fr:
            # --- importance-ordered ablation (necessity) -------------------
            keep = select_units_by_energy(energies, f, order="top")
            removed = exact_removed_energy(model, z_final, keep)
            out = model(x, n_steps=K,
                        intervene=make_ablation_hook(keep, None if cfg.persistent else K - 1, K))
            d_top = batch_dice(binarize(out.logits, cfg.threshold), y)

            # --- matched random null ---------------------------------------
            d_rand = np.zeros_like(d_top)
            rem_rand = torch.zeros_like(removed)
            for r in range(cfg.n_random):
                keep_r = select_units_by_energy(
                    energies, f, order="random", seed=cfg.seed * 1000 + r
                )
                rem_rand += exact_removed_energy(model, z_final, keep_r)
                out_r = model(x, n_steps=K,
                              intervene=make_ablation_hook(keep_r, None if cfg.persistent else K - 1, K))
                d_rand += batch_dice(binarize(out_r.logits, cfg.threshold), y)
            d_rand /= max(cfg.n_random, 1)
            rem_rand /= max(cfg.n_random, 1)

            # --- hardest contrast: same energy, taken from the smallest units
            keep_b = select_units_by_energy(energies, f, order="bottom")
            out_b = model(x, n_steps=K,
                          intervene=make_ablation_hook(keep_b, None if cfg.persistent else K - 1, K))
            d_bot = batch_dice(binarize(out_b.logits, cfg.threshold), y)

            # --- sufficiency: keep ONLY the top units, drop everything else -
            # select_units_by_energy marks the top-f units for ablation, so the
            # complement of its keep-mask is exactly "keep only the top f".
            keep_suff = 1.0 - select_units_by_energy(energies, f, order="top")
            out_s = model(x, n_steps=K,
                          intervene=make_ablation_hook(keep_suff, None if cfg.persistent else K - 1, K))
            d_suff = batch_dice(binarize(out_s.logits, cfg.threshold), y)

            for i, row in enumerate(rows):
                row[f"nec_top@{f}"] = float(base_dice[i] - d_top[i])
                row[f"nec_rand@{f}"] = float(base_dice[i] - d_rand[i])
                row[f"nec_bottom@{f}"] = float(base_dice[i] - d_bot[i])
                row[f"csi@{f}"] = float((base_dice[i] - d_top[i]) - (base_dice[i] - d_rand[i]))
                row[f"csi_tb@{f}"] = float(d_bot[i] - d_top[i])
                row[f"suff@{f}"] = float(d_suff[i] / max(base_dice[i], 1e-6))
                row[f"removed_top@{f}"] = float(removed[i])
                row[f"removed_rand@{f}"] = float(rem_rand[i])

        # --- naive top-1 necessity (confounded; reported for comparability) -
        top1 = torch.zeros_like(energies)
        top1.scatter_(1, energies.argmax(dim=1, keepdim=True), 1.0)
        keep1 = (1.0 - top1)[:, :, None, None]
        out1 = model(x, n_steps=K,
                     intervene=make_ablation_hook(keep1, None if cfg.persistent else K - 1, K))
        d1 = batch_dice(binarize(out1.logits, cfg.threshold), y)
        for i, row in enumerate(rows):
            row["naive_top1_necessity"] = float(base_dice[i] - d1[i])

        per_image.extend(rows)
        seen += x.shape[0]

    return {"per_image": per_image, "fractions": fr,
            "summary": summarize_curves(per_image, fr)}


def _auc(xs: Sequence[float], ys: Sequence[float]) -> float:
    """Trapezoidal AUC normalised by the x-range, so it reads as a mean."""
    x = np.asarray(xs, dtype=np.float64)
    y = np.asarray(ys, dtype=np.float64)
    if x.size < 2:
        return float(y.mean()) if y.size else float("nan")
    order = np.argsort(x)
    x, y = x[order], y[order]
    trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy >=2.0 renamed it
    return float(trapz(y, x) / (x[-1] - x[0]))


def summarize_curves(per_image: List[Dict[str, float]], fractions: Sequence[float]) -> Dict[str, float]:
    if not per_image:
        return {}
    def mean(key: str) -> float:
        v = np.array([r.get(key, np.nan) for r in per_image], dtype=np.float64)
        v = v[np.isfinite(v)]
        return float(v.mean()) if v.size else float("nan")

    nec_top = [mean(f"nec_top@{f}") for f in fractions]
    nec_rand = [mean(f"nec_rand@{f}") for f in fractions]
    nec_bot = [mean(f"nec_bottom@{f}") for f in fractions]
    csi = [mean(f"csi@{f}") for f in fractions]
    csi_tb = [mean(f"csi_tb@{f}") for f in fractions]
    suff = [mean(f"suff@{f}") for f in fractions]

    base = mean("base_dice")
    out: Dict[str, float] = {
        "base_dice": base,
        "n_active_units": mean("n_active_units"),
        "naive_top1_necessity": mean("naive_top1_necessity"),
        "necessity_auc": _auc(fractions, nec_top),
        "random_null_auc": _auc(fractions, nec_rand),
        "sufficiency_auc": _auc(fractions, suff),
        "bottom_ordered_auc": _auc(fractions, nec_bot),
        "CSI": _auc(fractions, csi),
        "CSI_normalized": _auc(fractions, csi) / max(base, 1e-6),
        "CSI_topbottom": _auc(fractions, csi_tb),
        "energy_match_ratio": mean(f"removed_top@{fractions[0]}") /
                              max(mean(f"removed_rand@{fractions[0]}"), 1e-9),
    }
    for f, a, b, bo, c, sf in zip(fractions, nec_top, nec_rand, nec_bot, csi, suff):
        out[f"nec_top@{f}"] = a
        out[f"nec_rand@{f}"] = b
        out[f"nec_bottom@{f}"] = bo
        out[f"csi@{f}"] = c
        out[f"suff@{f}"] = sf
    return out


def csi_vector(per_image: List[Dict[str, float]], fractions: Sequence[float]) -> np.ndarray:
    """Per-image CSI (AUC over rho) -- the vector fed to the paired tests."""
    return np.array(
        [_auc(fractions, [r.get(f"csi@{f}", np.nan) for f in fractions]) for r in per_image],
        dtype=np.float64,
    )


# --------------------------------------------------------------------------
# State transplantation
# --------------------------------------------------------------------------
@torch.no_grad()
def state_transplant(
    model: SPARCSeg,
    loader,
    device: torch.device,
    cfg: CausalConfig,
    step: Optional[int] = None,
) -> Dict[str, object]:
    """Interchange intervention: inject donor codes into a recipient's run.

    Three quantities per pair:
      ``toward_donor``  Dice(recipient-with-donor-code, donor GT)
                        - Dice(recipient, donor GT)     > 0 means the output
                        moved toward the donor's content.
      ``away_self``     Dice(recipient, own GT)
                        - Dice(recipient-with-donor-code, own GT)   >= 0.
      ``TTI``           toward_donor / (away_self + eps): transfer per unit of
                        damage. Random noise scores ~0 on this ratio; a genuinely
                        content-bearing state scores well above 0.
    """
    model.eval()
    K = model.n_steps
    t_inject = (K - 1) if step is None else step

    xs, ys, codes = [], [], []
    seen = 0
    for batch in loader:
        if seen >= cfg.max_images:
            break
        x = batch["image"].to(device)
        out = model(x, n_steps=K)
        xs.append(x.cpu())
        ys.append(batch["mask"].squeeze(1).numpy() > 0.5)
        codes.append(out.codes[-1].cpu())
        seen += x.shape[0]
    if not xs:
        return {"per_pair": [], "summary": {}}

    X = torch.cat(xs)
    Y = np.concatenate(ys)
    Z = torch.cat(codes)
    n = X.shape[0]

    rng = np.random.default_rng(cfg.seed)
    n_pairs = min(cfg.n_transplant_pairs, n * (n - 1))
    donors = rng.integers(0, n, size=n_pairs)
    recips = rng.integers(0, n, size=n_pairs)
    ok = donors != recips
    donors, recips = donors[ok], recips[ok]

    records: List[Dict[str, float]] = []
    bs = cfg.batch_size
    for start in range(0, len(donors), bs):
        d_idx = donors[start:start + bs]
        r_idx = recips[start:start + bs]
        xr = X[r_idx].to(device)
        zd = Z[d_idx].to(device)

        base = model(xr, n_steps=K)
        base_pred = binarize(base.logits, cfg.threshold)

        def hook(t: int, z: torch.Tensor, _zd=zd, _ti=t_inject) -> torch.Tensor:
            return _zd if t == _ti else z

        swapped = model(xr, n_steps=K, intervene=hook)
        swap_pred = binarize(swapped.logits, cfg.threshold)

        gt_d = Y[d_idx]
        gt_r = Y[r_idx]
        for i in range(len(d_idx)):
            toward = dice_score(swap_pred[i], gt_d[i]) - dice_score(base_pred[i], gt_d[i])
            away = dice_score(base_pred[i], gt_r[i]) - dice_score(swap_pred[i], gt_r[i])
            records.append({
                "toward_donor": float(toward),
                "away_self": float(away),
                "TTI": float(toward / (abs(away) + 1e-3)),
            })

    def m(k: str) -> float:
        v = np.array([r[k] for r in records], dtype=np.float64)
        v = v[np.isfinite(v)]
        return float(v.mean()) if v.size else float("nan")

    return {
        "per_pair": records,
        "summary": {"toward_donor": m("toward_donor"), "away_self": m("away_self"),
                    "TTI": m("TTI"), "n_pairs": float(len(records))},
    }


# --------------------------------------------------------------------------
# Steering
# --------------------------------------------------------------------------
def mask_properties(pred: np.ndarray) -> Dict[str, float]:
    """Measurable properties a steered atom might move. Deliberately simple and
    fully objective -- these are what atom names are validated against."""
    area = float(pred.sum())
    if area == 0:
        return {"area": 0.0, "perimeter": 0.0, "compactness": float("nan"),
                "boundary_length_ratio": float("nan")}
    per = 0.0
    per += float((pred[:, 1:] != pred[:, :-1]).sum())
    per += float((pred[1:, :] != pred[:-1, :]).sum())
    return {
        "area": area,
        "perimeter": per,
        "compactness": float(4 * np.pi * area / (per ** 2 + 1e-8)),
        "boundary_length_ratio": float(per / (np.sqrt(area) + 1e-8)),
    }


@torch.no_grad()
def steering_test(
    model: SPARCSeg,
    loader,
    device: torch.device,
    cfg: CausalConfig,
    atom_ids: Optional[Sequence[int]] = None,
) -> Dict[str, object]:
    """Scale one atom's coefficients by alpha and track output properties.

    A *monotone* response (Spearman |rho| high across the alpha grid) is the
    directed causal evidence that ablation alone cannot provide: breaking a
    component proves it was used; steering it proves what it was used *for*.
    """
    pass  # (inlined below)

    model.eval()
    K = model.n_steps
    alphas = list(cfg.steering_alphas)

    # Pick the most-used atoms if none specified.
    if atom_ids is None:
        mass = torch.zeros(model.dictionary.m, device=device)
        seen = 0
        for batch in loader:
            if seen >= cfg.max_images:
                break
            x = batch["image"].to(device)
            out = model(x, n_steps=K)
            mass += unit_energy(out.codes[-1]).sum(0)
            seen += x.shape[0]
        atom_ids = torch.topk(mass, k=min(cfg.top_atoms_for_steering,
                                          model.dictionary.m)).indices.tolist()

    results: Dict[int, Dict[str, float]] = {}
    for j in atom_ids:
        curves: Dict[str, List[List[float]]] = {k: [] for k in
                                                ("area", "perimeter", "compactness",
                                                 "boundary_length_ratio")}
        seen = 0
        for batch in loader:
            if seen >= min(cfg.max_images, 128):
                break
            x = batch["image"].to(device)
            per_alpha: Dict[str, List[List[float]]] = {k: [] for k in curves}
            for a in alphas:
                scale = torch.ones(model.dictionary.m, device=device)
                scale[j] = a

                def hook(t: int, z: torch.Tensor, _s=scale) -> torch.Tensor:
                    return z * _s[None, :, None, None]

                out = model(x, n_steps=K, intervene=hook)
                preds = binarize(out.logits, cfg.threshold)
                props = [mask_properties(p) for p in preds]
                for k in curves:
                    per_alpha[k].append([p[k] for p in props])
            for k in curves:
                arr = np.array(per_alpha[k], dtype=np.float64)   # (n_alpha, B)
                curves[k].extend(arr.T.tolist())
            seen += x.shape[0]

        entry: Dict[str, float] = {}
        for k, rows in curves.items():
            rhos = [spearman(np.array(alphas), np.array(r))["rho"] for r in rows]
            rhos = [r for r in rhos if np.isfinite(r)]
            entry[f"{k}_rho"] = float(np.mean(rhos)) if rhos else float("nan")
            entry[f"{k}_abs_rho"] = float(np.mean(np.abs(rhos))) if rhos else float("nan")
        results[int(j)] = entry

    finite = [v["area_abs_rho"] for v in results.values() if np.isfinite(v.get("area_abs_rho", np.nan))]
    return {
        "per_atom": results,
        "summary": {"mean_area_abs_rho": float(np.mean(finite)) if finite else float("nan"),
                    "n_atoms": float(len(results))},
    }


# --------------------------------------------------------------------------
# Spatial alignment of an atom with the change it causes
# --------------------------------------------------------------------------
@torch.no_grad()
def spatial_alignment(
    model: SPARCSeg, loader, device: torch.device, cfg: CausalConfig,
    n_atoms: int = 8,
) -> Dict[str, float]:
    """IoU between where an atom is active and where ablating it changes the mask.

    High alignment means the explanation is *local*, not merely global: the atom
    is doing work in the region it lights up in.  A dense state's channels light
    up everywhere and score near chance, which is the point of the comparison.
    """
    model.eval()
    K = model.n_steps
    scores: List[float] = []
    chance: List[float] = []
    seen = 0

    for batch in loader:
        if seen >= min(cfg.max_images, 128):
            break
        x = batch["image"].to(device)
        base = model(x, n_steps=K)
        z = base.codes[-1]
        base_pred = binarize(base.logits, cfg.threshold)
        energies = unit_energy(z)
        top = torch.topk(energies, k=min(n_atoms, energies.shape[1]), dim=1).indices

        for slot in range(top.shape[1]):
            keep = torch.ones_like(energies)
            keep.scatter_(1, top[:, slot:slot + 1], 0.0)
            out = model(x, n_steps=K,
                        intervene=make_ablation_hook(keep[:, :, None, None],
                                                     None if cfg.persistent else K - 1, K))
            pred = binarize(out.logits, cfg.threshold)
            changed = np.logical_xor(pred, base_pred)

            for b in range(x.shape[0]):
                j = int(top[b, slot])
                act = z[b, j].cpu().numpy()
                if act.max() <= 0:
                    continue
                act_up = np.array(
                    F.interpolate(torch.tensor(act)[None, None], size=changed.shape[-2:],
                                  mode="bilinear", align_corners=False)[0, 0]
                )
                support = act_up > (0.25 * act_up.max())
                ch = changed[b]
                if not ch.any() or not support.any():
                    continue
                inter = np.logical_and(support, ch).sum()
                union = np.logical_or(support, ch).sum()
                scores.append(float(inter / max(union, 1)))
                # chance: same-size support placed at a random offset
                shift = np.roll(support, shift=(support.shape[0] // 3, support.shape[1] // 3),
                                axis=(0, 1))
                chance.append(float(np.logical_and(shift, ch).sum() /
                                    max(np.logical_or(shift, ch).sum(), 1)))
        seen += x.shape[0]

    if not scores:
        return {"alignment_iou": float("nan"), "chance_iou": float("nan"),
                "alignment_gain": float("nan"), "n": 0.0}
    a, c = float(np.mean(scores)), float(np.mean(chance))
    return {"alignment_iou": a, "chance_iou": c, "alignment_gain": a - c,
            "n": float(len(scores))}


# --------------------------------------------------------------------------
# Is the code a real bottleneck on the sketch?
# --------------------------------------------------------------------------
@torch.no_grad()
def bottleneck_diagnostics(model: SPARCSeg, loader, device: torch.device,
                           cfg: CausalConfig) -> Dict[str, float]:
    """How much of the final sketch the code explains, and how sparse it is.

    This is the structural precondition for every causal result in this file.
    If ``code_explained_variance`` is near zero, the readout is decoding a
    sketch the code did not build, no intervention on z can matter, and a high
    CSI would be impossible rather than merely absent.  Publishing the causal
    table without this number invites the obvious objection that the
    intervention was simply too weak to register.
    """
    model.eval()
    ev, active, resid_to_evidence = [], [], []
    seen = 0
    for batch in loader:
        if seen >= cfg.max_images:
            break
        x = batch["image"].to(device)
        out = model(x, n_steps=model.n_steps)
        z, s_final = out.codes[-1], out.sketches[-1]
        ev.extend(model.code_explained_variance(z, s_final).cpu().tolist())
        active.extend(model.dictionary.usage_stats(z)["atoms_active_per_image"].cpu().tolist())
        drift = ((s_final - out.evidence).pow(2).flatten(1).sum(1) /
                 out.evidence.pow(2).flatten(1).sum(1).clamp_min(1e-8))
        resid_to_evidence.extend(drift.cpu().tolist())
        seen += x.shape[0]
    return {
        "code_explained_variance": float(np.mean(ev)) if ev else float("nan"),
        "atoms_active_per_image": float(np.mean(active)) if active else float("nan"),
        "sketch_drift_from_evidence": float(np.mean(resid_to_evidence)) if resid_to_evidence else float("nan"),
    }


# --------------------------------------------------------------------------
# Does the energy actually track the answer?
# --------------------------------------------------------------------------
@torch.no_grad()
def energy_error_coupling(model, loader, device: torch.device,
                          cfg: CausalConfig) -> Dict[str, float]:
    """Correlate per-step energy change with per-step Dice change.

    This is the honest answer to the obvious objection: a monotone energy is a
    property of the optimiser, not evidence that the *prediction* improves.  If
    the two are uncorrelated, the convergence guarantee is decorative -- exactly
    the charge the workshop levels at latent reasoning in general -- so the paper
    is better off measuring it than asserting it.
    """
    pass  # (inlined below)

    model.eval()
    d_e: List[float] = []
    d_dice: List[float] = []
    mono: List[float] = []
    seen = 0

    for batch in loader:
        if seen >= cfg.max_images:
            break
        x = batch["image"].to(device)
        y = batch["mask"].squeeze(1).numpy() > 0.5
        out = model(x, n_steps=model.n_steps)
        if out.energy is None or len(out.logits_per_step) < 2:
            break
        E = out.energy.stack().numpy()                       # (K+1, B)
        dices = np.stack([batch_dice(binarize(lg, cfg.threshold), y)
                          for lg in out.logits_per_step])     # (K, B)
        for b in range(x.shape[0]):
            for t in range(dices.shape[0] - 1):
                d_e.append(float(E[t + 2, b] - E[t + 1, b]))
                d_dice.append(float(dices[t + 1, b] - dices[t, b]))
        mono.extend(out.energy.is_monotone().float().tolist())
        seen += x.shape[0]

    # Reported as descent-vs-improvement so the sign reads the intuitive way:
    # POSITIVE means "steps that lower the energy more also improve Dice more".
    sp = spearman(-np.array(d_e), np.array(d_dice))
    return {
        "energy_gain_alignment": sp["rho"],
        "spearman_dE_dDice": -sp["rho"] if np.isfinite(sp["rho"]) else float("nan"),
        "p": sp["p"],
        "n": float(sp["n"]),
        "monotone_descent_rate": float(np.mean(mono)) if mono else float("nan"),
    }

---
## Part 4 - Training and evaluation

One loop serves every method, so optimiser, schedule, augmentation, epoch count and early-stopping rule are provably identical across rows of the results table.

In [ ]:
# ===== sparcseg/train.py =====================================================
"""Training, evaluation and the efficiency measurements.

One loop serves every method so that optimiser, schedule, augmentation, epoch
count and early-stopping rule are provably identical across rows of the results
table.  Method-specific behaviour is confined to two small hooks
(``_extra_loss`` and ``model.on_optimizer_step``).
"""

from __future__ import annotations

import copy
import math
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)


# --------------------------------------------------------------------------
# Schedule
# --------------------------------------------------------------------------
def cosine_warmup(optimizer, total_steps: int, warmup_steps: int):
    def fn(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / max(warmup_steps, 1)
        prog = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def _extra_loss(model: nn.Module, batch_x: torch.Tensor,
                batch_y: torch.Tensor) -> torch.Tensor:
    """PTEA-lite needs its plausibility energy trained; everything else is 0."""
    if isinstance(model, PTEALite):
        return model.energy_training_loss(batch_x, batch_y)
    return batch_x.new_zeros(())


# --------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader,
    device: torch.device,
    cfg,
    adaptive: Optional[bool] = None,
    n_steps: Optional[int] = None,
    collect_ids: bool = True,
) -> Dict[str, object]:
    model.eval()
    per_image: List[Dict[str, float]] = []
    ids: List[str] = []
    steps_used: List[float] = []
    monotone: List[float] = []

    use_adaptive = cfg.adaptive_depth if adaptive is None else adaptive
    kw: Dict[str, object] = {}
    if isinstance(model, SPARCSeg):
        kw = dict(
            # Adaptive eval caps at the TRAINED depth. Running deeper than the
            # network was ever optimised for is extrapolation, and reporting it
            # as the headline number would confound early exit with depth
            # generalisation. ``efficiency_sweep`` probes beyond K separately
            # and labels it as such.
            n_steps=n_steps or cfg.n_steps,
            adaptive=use_adaptive,
            plateau_eps=cfg.energy_plateau_eps,
            min_steps=cfg.min_steps,
            backtracking=cfg.backtracking_eval,
            backtrack_shrink=cfg.backtrack_shrink,
            backtrack_max=cfg.backtrack_max,
            armijo_c=cfg.armijo_c,
        )

    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].squeeze(1).numpy() > 0.5
        out = model(x, **kw)
        pred = (torch.sigmoid(out.logits) > cfg.prob_threshold).squeeze(1).cpu().numpy()
        for i in range(x.shape[0]):
            per_image.append(all_metrics(pred[i], y[i], cfg.boundary_tolerances))
        if collect_ids:
            ids.extend(batch["sample_id"])
        if out.steps_used is not None:
            steps_used.extend(out.steps_used.detach().cpu().tolist())
        if out.energy is not None and out.energy.values:
            monotone.extend(out.energy.is_monotone().float().tolist())

    agg = aggregate(per_image)
    non_empty_gt = [d for d in per_image if not d.get("gt_empty", 0.0)]
    if non_empty_gt and all(d["pred_area"] == 0 for d in non_empty_gt):
        agg["collapsed_to_empty"] = 1.0
        # Printed only for test-set evaluations (collect_ids=True). The depth
        # sweep calls evaluate() a dozen times per model, and a warning repeated
        # a dozen times per fold buries the output it is meant to draw attention
        # to. The flag itself is always set, so nothing is lost in results.json.
        if collect_ids:
            print("  [warn] model predicts an EMPTY mask on every lesion-bearing "
                  "image. Dice here is driven entirely by the empty-GT cases and "
                  "is not comparable. Usual causes: too few epochs, or pos_weight "
                  "too low.")
    if steps_used:
        agg["mean_steps"] = float(np.mean(steps_used))
    if monotone:
        agg["monotone_descent_rate"] = float(np.mean(monotone))
    return {"per_image": per_image, "aggregate": agg, "ids": ids}


# --------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------
@dataclass
class TrainResult:
    state_dict: Dict
    history: List[Dict[str, float]]
    best_val: float
    best_epoch: int
    train_seconds: float


def train_model(
    model: nn.Module,
    train_loader,
    val_loader,
    cfg,
    device: torch.device,
    method: str = "sparcseg",
    verbose: bool = True,
    epochs: Optional[int] = None,
) -> TrainResult:
    model.to(device)
    n_epochs = epochs or cfg.epochs

    optimizer = torch.optim.AdamW(
        model.param_groups(cfg.lr, cfg.lr_backbone_mult, cfg.weight_decay)
    )
    steps_per_epoch = max(len(train_loader), 1)
    scheduler = cosine_warmup(optimizer, n_epochs * steps_per_epoch,
                              cfg.warmup_epochs * steps_per_epoch)
    use_amp = bool(cfg.amp and device.type == "cuda")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        autocast = lambda: torch.amp.autocast("cuda", enabled=use_amp)
    except (AttributeError, TypeError):  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        autocast = lambda: torch.cuda.amp.autocast(enabled=use_amp)

    pos_weight = positive_weight_from_loader(train_loader, device=device)
    criterion = SPARCSegLoss(
        alpha=cfg.dice_ce_alpha,
        deep_decay=cfg.deep_supervision_decay,
        deep_supervision=cfg.deep_supervision,
    )

    best_val, best_epoch = -1.0, -1
    best_state = copy.deepcopy(model.state_dict())
    history: List[Dict[str, float]] = []
    t0 = time.time()

    for epoch in range(n_epochs):
        model.train()
        loss_meter, seg_meter = AverageMeter(), AverageMeter()
        for batch in train_loader:
            x = batch["image"].to(device, non_blocking=True)
            y = batch["mask"].to(device, non_blocking=True)

            fwd: Dict[str, object] = {}
            if isinstance(model, SPARCSeg) and getattr(cfg, "train_depth_sampling", False):
                # Uniform over {1..K}: makes the network depth-robust, which is
                # what turns early exit into a real property rather than an
                # artefact of always unrolling exactly K times.
                fwd["n_steps"] = int(torch.randint(1, cfg.n_steps + 1, (1,)).item())

            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = model(x, **fwd)
                parts = criterion(out, y, pos_weight=pos_weight)
                loss = parts["total"] + _extra_loss(model, x, y)

            scaler.scale(loss).backward()
            if cfg.grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            model.on_optimizer_step()

            loss_meter.update(float(loss.detach()), x.shape[0])
            seg_meter.update(float(parts["seg"].detach()), x.shape[0])

        val = evaluate(model, val_loader, device, cfg, collect_ids=False)
        val_dice = val["aggregate"].get("dice", float("nan"))
        history.append({
            "epoch": epoch, "loss": loss_meter.avg, "seg": seg_meter.avg,
            "val_dice": val_dice, "val_bf2": val["aggregate"].get("bf2", float("nan")),
            "lr": optimizer.param_groups[0]["lr"],
        })

        if val_dice > best_val:
            best_val, best_epoch = val_dice, epoch
            best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch % 5 == 0 or epoch == n_epochs - 1):
            print(f"  [{method:>16}] epoch {epoch:3d}/{n_epochs}  "
                  f"loss {loss_meter.avg:.4f}  val_dice {val_dice:.4f}  "
                  f"(best {best_val:.4f} @ {best_epoch})")

    model.load_state_dict(best_state)
    return TrainResult(best_state, history, best_val, best_epoch, time.time() - t0)


# --------------------------------------------------------------------------
# Efficiency: the adaptive-depth result
# --------------------------------------------------------------------------
@torch.no_grad()
def efficiency_sweep(
    model: nn.Module,
    loader,
    device: torch.device,
    cfg,
    fixed_depths: Sequence[int] = (1, 2, 3, 4, 6, 8),
) -> Dict[str, object]:
    """Accuracy-vs-compute curve: fixed K sweep against adaptive early exit.

    Compute is reported as *mean reasoning steps actually executed* and as
    wall-clock ms/image measured on this device.  Step count alone would be
    misleading -- adaptive depth adds an energy evaluation per step -- so both
    are reported and the paper should quote the wall-clock one.
    """
    if not isinstance(model, SPARCSeg):
        return {}
    model.eval()
    rows: List[Dict[str, float]] = []

    def timed_eval(**kw) -> Tuple[Dict[str, float], float]:
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        res = evaluate(model, loader, device, cfg, collect_ids=False, **kw)
        if device.type == "cuda":
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        n = max(res["aggregate"].get("n_images", 1.0), 1.0)
        return res["aggregate"], 1000.0 * dt / n

    for k in fixed_depths:
        agg, ms = timed_eval(adaptive=False, n_steps=k)
        mode = "fixed" if k <= cfg.n_steps else "fixed (extrapolated)"
        rows.append({"mode": mode, "K": float(k), "mean_steps": float(k),
                     "dice": agg.get("dice", float("nan")),
                     "bf2": agg.get("bf2", float("nan")),
                     "ms_per_image": ms})

    agg, ms = timed_eval(adaptive=True, n_steps=cfg.n_steps)
    rows.append({"mode": "adaptive", "K": float(cfg.n_steps),
                 "mean_steps": agg.get("mean_steps", float("nan")),
                 "dice": agg.get("dice", float("nan")),
                 "bf2": agg.get("bf2", float("nan")),
                 "ms_per_image": ms})

    fixed = [r for r in rows if r["mode"] == "fixed"]  # trained depths only
    ad = rows[-1]
    matched = [r for r in fixed if r["dice"] >= ad["dice"] - 0.002]
    speedup = (min(r["mean_steps"] for r in matched) / max(ad["mean_steps"], 1e-6)
               if matched else float("nan"))
    return {"rows": rows,
            "summary": {"adaptive_mean_steps": ad["mean_steps"],
                        "adaptive_dice": ad["dice"],
                        "step_saving_vs_matched_fixed": speedup}}


@torch.no_grad()
def per_step_accuracy(model: nn.Module, loader, device: torch.device,
                      cfg) -> List[Dict[str, float]]:
    """Dice/BF at each reasoning step -- the 'does revision actually revise?' plot."""
    if not isinstance(model, SPARCSeg):
        return []
    model.eval()
    acc: Dict[int, List[Dict[str, float]]] = {}
    for batch in loader:
        x = batch["image"].to(device)
        y = batch["mask"].squeeze(1).numpy() > 0.5
        out = model(x, n_steps=cfg.n_steps, adaptive=False)
        for t, lg in enumerate(out.logits_per_step):
            pred = (torch.sigmoid(lg) > cfg.prob_threshold).squeeze(1).cpu().numpy()
            acc.setdefault(t, []).extend(
                all_metrics(pred[i], y[i], cfg.boundary_tolerances, with_topology=False)
                for i in range(x.shape[0])
            )
    return [{"step": float(t + 1), **aggregate(v)} for t, v in sorted(acc.items())]

In [ ]:
# ===== sparcseg/experiment.py ================================================
"""End-to-end experiment driver: every table and figure the paper needs.

A full run produces, per dataset:

  T1  main results     -- 5 methods x {Dice, IoU, BF@2, BF@5, HD95, Betti err}
                          with paired Wilcoxon + Holm against SPARC-Seg
  T2  faithfulness     -- CSI, necessity/sufficiency AUC, naive top-1 (labelled
                          confounded), transplant TTI, steering rho, spatial
                          alignment; SPARC-Seg vs the dense control
  T3  efficiency       -- accuracy/compute curve, adaptive vs fixed depth
  T4  concept vocabulary and naming stability
  T5  ablations        -- m sweep, K sweep, topo on/off, group-sparsity on/off,
                          usage-balance on/off, low-label regime
  D1  diagnostics      -- monotone-descent rate, energy-vs-Dice coupling,
                          dictionary coherence, dead-atom count

Runtime is controlled by ``ExperimentPlan``.  ``quick_plan()`` fits comfortably
inside a single Kaggle T4 session; ``full_plan()`` is the paper configuration
and is meant to be split across sessions using ``only_methods`` / ``folds``.
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch

pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)
pass  # (inlined below)


# --------------------------------------------------------------------------
# Plans
# --------------------------------------------------------------------------
@dataclass
class ExperimentPlan:
    methods: Tuple[str, ...] = ("singleshot", "dense_unrolled", "ptea_lite",
                                "textual_bottleneck", "sparcseg")
    folds: Tuple[int, ...] = (0,)
    seeds: Tuple[int, ...] = (0,)
    epochs: Optional[int] = None
    run_causal: bool = True
    run_transplant: bool = True
    run_steering: bool = True
    run_alignment: bool = True
    run_concepts: bool = True
    run_efficiency: bool = True
    run_per_step: bool = True
    run_ablations: bool = False
    low_label_fracs: Tuple[float, ...] = ()
    dict_sizes: Tuple[int, ...] = ()
    step_counts: Tuple[int, ...] = ()
    topk_values: Tuple[int, ...] = ()
    max_train_images: Optional[int] = None
    save_checkpoints: bool = False


def quick_plan(epochs: int = 12) -> ExperimentPlan:
    """~45-75 min on a T4 for BUSI. Use this to smoke-test end to end first."""
    return ExperimentPlan(epochs=epochs, folds=(0,), run_ablations=False,
                          run_steering=False, max_train_images=None)


def full_plan() -> ExperimentPlan:
    """The paper configuration. Expect several Kaggle sessions per dataset."""
    return ExperimentPlan(
        folds=(0, 1, 2, 3, 4),
        seeds=(0, 1, 2),
        run_ablations=True,
        low_label_fracs=(0.1, 0.25, 0.5),
        dict_sizes=(64, 128, 192, 256),
        step_counts=(1, 2, 3, 4, 6),
        topk_values=(2, 4, 8, 16, 32),
    )


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
def load_samples(dataset_key: str, root: Optional[str] = None):
    # Uniquely named per dataset: the notebook build flattens every module into
    # one namespace, where three functions called ``build_index`` would silently
    # shadow each other and every notebook would load whichever came last.
    pass  # (inlined below)
    pass  # (inlined below)
    pass  # (inlined below)

    builders = {"busi": build_index_busi, "isic": build_index_isic,
                "brisc": build_index_brisc}
    if dataset_key not in builders:
        raise ValueError(f"unknown dataset {dataset_key!r}")
    return builders[dataset_key](root)


def build_fold_loaders(samples, fold: int, cfg: CoreConfig, seed: int,
                       label_frac: float = 1.0,
                       max_train_images: Optional[int] = None):
    folds = stratified_folds(samples, cfg.n_folds, seed=0)  # split seed fixed
    test_idx = folds[fold]
    train_pool = np.setdiff1d(np.arange(len(samples)), test_idx)
    train_idx, val_idx = split_train_val(train_pool, samples,
                                         cfg.val_frac_within_train, seed=0)
    if label_frac < 1.0:
        train_idx = label_subset(train_idx, samples, label_frac, seed=seed)
    if max_train_images is not None and len(train_idx) > max_train_images:
        rng = np.random.default_rng(seed)
        train_idx = np.sort(rng.choice(train_idx, max_train_images, replace=False))

    pick = lambda idx: [samples[i] for i in idx]
    ds_tr = SegmentationDataset(pick(train_idx), cfg.img_size, True, seed=seed)
    ds_va = SegmentationDataset(pick(val_idx), cfg.img_size, False, seed=seed)
    ds_te = SegmentationDataset(pick(test_idx), cfg.img_size, False, seed=seed)
    return (
        make_loader(ds_tr, cfg.batch_size, True, cfg.num_workers, drop_last=True),
        make_loader(ds_va, cfg.batch_size, False, cfg.num_workers),
        make_loader(ds_te, cfg.batch_size, False, cfg.num_workers),
        {"n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx)},
    )


# --------------------------------------------------------------------------
# One (method, fold, seed) cell
# --------------------------------------------------------------------------
def run_cell(method: str, dataset_key: str, samples, cfg: CoreConfig, plan: ExperimentPlan,
             fold: int, seed: int, device: torch.device,
             label_frac: float = 1.0, verbose: bool = True) -> Dict[str, object]:
    set_seed(seed)
    tr, va, te, counts = build_fold_loaders(samples, fold, cfg, seed, label_frac,
                                            plan.max_train_images)
    model = build_model(method, cfg, in_channels=DATASETS[dataset_key].in_channels)
    n_params = count_params(model)
    res = train_model(model, tr, va, cfg, device, method=method,
                      verbose=verbose, epochs=plan.epochs)
    test = evaluate(model, te, device, cfg)
    if verbose:
        a = test["aggregate"]
        print(f"  -> {method:>18}  test Dice {a.get('dice', float('nan')):.4f}  "
              f"BF@2 {a.get('bf2', float('nan')):.4f}  ({human_time(res.train_seconds)})")
    return {"method": method, "fold": fold, "seed": seed, "label_frac": label_frac,
            "counts": counts, "n_params": n_params,
            "history": res.history, "best_val": res.best_val,
            "train_seconds": res.train_seconds,
            "test": test, "model": model, "loaders": (tr, va, te)}


# --------------------------------------------------------------------------
# Faithfulness block
# --------------------------------------------------------------------------
def run_faithfulness(cells: Dict[str, Dict], cfg: CoreConfig, plan: ExperimentPlan,
                     device: torch.device, verbose: bool = True) -> Dict[str, object]:
    """The paper's centerpiece: SPARC-Seg vs the dense control, same protocol."""
    ccfg = CausalConfig(
        fractions=cfg.ablation_fractions,
        n_random=cfg.n_random_controls,
        max_images=cfg.causal_max_images,
        batch_size=cfg.batch_size,
        steering_alphas=cfg.steering_alphas,
        n_transplant_pairs=cfg.n_transplant_pairs,
        threshold=cfg.prob_threshold,
    )
    out: Dict[str, object] = {"config": ccfg.__dict__}
    per_method_csi: Dict[str, np.ndarray] = {}

    for method in ("sparcseg", "dense_unrolled"):
        cell = cells.get(method)
        if cell is None or not isinstance(cell["model"], SPARCSeg):
            continue
        model, te = cell["model"], cell["loaders"][2]
        block: Dict[str, object] = {}

        block["bottleneck"] = bottleneck_diagnostics(model, te, device, ccfg)
        if verbose:
            b = block["bottleneck"]
            print(f"  [faithfulness] {method}: code explains "
                  f"{b['code_explained_variance']:.1%} of the sketch, "
                  f"{b['atoms_active_per_image']:.1f} atoms active/image")
            print(f"  [faithfulness] {method}: norm-matched ablation curves ...")
        curves = ablation_curves(model, te, device, ccfg)
        block["curves"] = curves["summary"]
        block["fractions"] = curves["fractions"]
        per_method_csi[method] = csi_vector(curves["per_image"], curves["fractions"])

        if plan.run_transplant:
            if verbose:
                print(f"  [faithfulness] {method}: state transplantation ...")
            block["transplant"] = state_transplant(model, te, device, ccfg)["summary"]

        if plan.run_steering:
            if verbose:
                print(f"  [faithfulness] {method}: steering ...")
            block["steering"] = steering_test(model, te, device, ccfg)["summary"]

        if plan.run_alignment:
            if verbose:
                print(f"  [faithfulness] {method}: spatial alignment ...")
            block["alignment"] = spatial_alignment(model, te, device, ccfg)

        block["energy_coupling"] = energy_error_coupling(model, te, device, ccfg)
        block["dict_coherence"] = float(model.dictionary.coherence())
        out[method] = block

    if len(per_method_csi) == 2:
        a, b = per_method_csi["sparcseg"], per_method_csi["dense_unrolled"]
        n = min(len(a), len(b))
        out["csi_comparison"] = {
            **wilcoxon(a[:n], b[:n]),
            "sparcseg_csi": float(np.nanmean(a)),
            "dense_csi": float(np.nanmean(b)),
            "gap": float(np.nanmean(a) - np.nanmean(b)),
        }
    return out


# --------------------------------------------------------------------------
# Ablations
# --------------------------------------------------------------------------
def run_ablations(dataset_key: str, samples, cfg: CoreConfig, plan: ExperimentPlan,
                  device: torch.device, fold: int = 0, seed: int = 0,
                  verbose: bool = True) -> Dict[str, object]:
    rows: List[Dict[str, object]] = []

    def one(tag: str, variant_cfg: CoreConfig, method: str = "sparcseg",
            label_frac: float = 1.0) -> None:
        cell = run_cell(method, dataset_key, samples, variant_cfg, plan, fold, seed,
                        device, label_frac=label_frac, verbose=False)
        a = cell["test"]["aggregate"]
        rows.append({"ablation": tag, "dice": a.get("dice"), "bf2": a.get("bf2"),
                     "iou": a.get("iou"), "hd95": a.get("hd95"),
                     "betti0_err": a.get("betti0_err"),
                     "mean_steps": a.get("mean_steps"),
                     "n_params": cell["n_params"], "label_frac": label_frac})
        if verbose:
            print(f"    ablation {tag:<28} Dice {a.get('dice', float('nan')):.4f}")
        del cell

    one("reference", cfg)
    for m in plan.dict_sizes:
        one(f"dict_size={m}", cfg.replace(dict_size=m))
    for k in plan.step_counts:
        one(f"K={k}", cfg.replace(n_steps=k))
    for k in plan.topk_values:
        one(f"topk_atoms={k}", cfg.replace(topk_atoms=k))
    one("no_topo (lambda2=0)", cfg.replace(lambda_topo=0.0))
    one("no_evidence_term (lambda3=0)", cfg.replace(lambda_evidence=0.0))
    # Sparsity-mechanism ablations, matched to whichever mode is in use.
    if cfg.sparsity_mode == "topk":
        one("l1 penalty instead of top-k", cfg.replace(sparsity_mode="l1"))
        one("no straight-through", cfg.replace(straight_through=False))
    else:
        one("no_group_sparsity", cfg.replace(lambda_group=0.0))
        one("top-k instead of l1", cfg.replace(sparsity_mode="topk"))
    one("signed_code (nonneg off)", cfg.replace(nonneg_code=False))
    one("no_deep_supervision", cfg.replace(deep_supervision=False))
    for f in plan.low_label_fracs:
        one(f"labels={int(f * 100)}%", cfg, label_frac=f)
        one(f"labels={int(f * 100)}% [singleshot]", cfg, method="singleshot", label_frac=f)
    return {"rows": rows}


# --------------------------------------------------------------------------
# Driver
# --------------------------------------------------------------------------
def run_dataset_experiment(
    dataset_key: str,
    cfg: Optional[CoreConfig] = None,
    plan: Optional[ExperimentPlan] = None,
    root: Optional[str] = None,
    out_dir: str = "/kaggle/working/sparcseg_results",
    verbose: bool = True,
) -> Dict[str, object]:
    cfg = cfg or CoreConfig()
    plan = plan or quick_plan()
    device = get_device()
    dinfo = DATASETS[dataset_key]

    banner(f"SPARC-Seg  |  {dinfo.name}  |  {dinfo.modality} ({dinfo.physics})")
    samples = load_samples(dataset_key, root)
    print(f"dataset: {summarize_samples(samples)}")
    print(f"device : {device}  |  methods: {list(plan.methods)}  "
          f"|  folds: {list(plan.folds)}  seeds: {list(plan.seeds)}")

    out_path = Path(out_dir) / dataset_key
    out_path.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    results: Dict[str, object] = {
        "dataset": dataset_key,
        "dataset_info": {"name": dinfo.name, "modality": dinfo.modality,
                         "physics": dinfo.physics, "notes": dinfo.notes},
        "config": cfg.__dict__,
        "plan": plan.__dict__,
        "samples": summarize_samples(samples),
        "cells": [],
        "concepts": {},
    }

    last_cells: Dict[str, Dict] = {}
    per_method_dice: Dict[str, List[np.ndarray]] = {}
    concept_profiles_by_fold: Dict[int, List[Dict]] = {}

    for seed in plan.seeds:
        for fold in plan.folds:
            banner(f"fold {fold}  seed {seed}", char="-")
            cells: Dict[str, Dict] = {}
            for method in plan.methods:
                cell = run_cell(method, dataset_key, samples, cfg, plan, fold, seed,
                                device, verbose=verbose)
                cells[method] = cell
                per_method_dice.setdefault(method, []).append(
                    column(cell["test"]["per_image"], "dice")
                )
                results["cells"].append({
                    k: v for k, v in cell.items() if k not in {"model", "loaders"}
                } | {"test": {"aggregate": cell["test"]["aggregate"]}})

            # Faithfulness / concepts / efficiency use the most recent fold's models.
            if plan.run_causal:
                results.setdefault("faithfulness", {})[f"fold{fold}_seed{seed}"] = \
                    run_faithfulness(cells, cfg, plan, device, verbose)

            sp = cells.get("sparcseg")
            if sp is not None:
                if plan.run_concepts:
                    if verbose:
                        print("  [concepts] profiling atoms ...")
                    prof = profile_atoms(sp["model"], sp["loaders"][2], device,
                                         max_images=min(cfg.causal_max_images, 200))
                    concept_profiles_by_fold[fold] = prof["profiles"]
                    results["concepts"][f"fold{fold}_seed{seed}"] = {
                        "summary": prof["summary"], "top": top_atoms_table(prof)
                    }
                if plan.run_efficiency:
                    if verbose:
                        print("  [efficiency] depth sweep ...")
                    results.setdefault("efficiency", {})[f"fold{fold}_seed{seed}"] = \
                        efficiency_sweep(sp["model"], sp["loaders"][2], device, cfg)
                if plan.run_per_step:
                    results.setdefault("per_step", {})[f"fold{fold}_seed{seed}"] = \
                        per_step_accuracy(sp["model"], sp["loaders"][2], device, cfg)
                if plan.save_checkpoints:
                    torch.save(sp["model"].state_dict(),
                               out_path / f"sparcseg_fold{fold}_seed{seed}.pt")

            last_cells = cells
            for m, c in cells.items():
                c["model"] = c["model"].cpu()
            if device.type == "cuda":
                torch.cuda.empty_cache()

    # ---- pooled statistics across folds/seeds -----------------------------
    pooled = {m: np.concatenate(v) for m, v in per_method_dice.items() if v}
    if "sparcseg" in pooled and len(pooled) > 1:
        n = min(len(v) for v in pooled.values())
        results["significance"] = compare_methods(
            {m: v[:n] for m, v in pooled.items()}, reference="sparcseg",
            n_boot=cfg.bootstrap_n
        )
    results["main_table"] = build_main_table(results["cells"])

    if len(concept_profiles_by_fold) > 1:
        keys = sorted(concept_profiles_by_fold)
        results["concepts"]["stability"] = float(np.nanmean([
            naming_stability(concept_profiles_by_fold[keys[i]],
                             concept_profiles_by_fold[keys[i + 1]])
            for i in range(len(keys) - 1)
        ]))

    if plan.run_ablations:
        banner("ablations", char="-")
        results["ablations"] = run_ablations(dataset_key, samples, cfg, plan, device,
                                             fold=plan.folds[0], seed=plan.seeds[0],
                                             verbose=verbose)

    results["wall_clock_seconds"] = time.time() - t_start
    save_json(results, out_path / "results.json")
    print(f"\nsaved -> {out_path / 'results.json'}   "
          f"(total {human_time(results['wall_clock_seconds'])})")
    results["_last_cells"] = last_cells
    return results


# --------------------------------------------------------------------------
# Table rendering
# --------------------------------------------------------------------------
def build_main_table(cells: Sequence[Dict]) -> List[Dict[str, object]]:
    by_method: Dict[str, List[Dict[str, float]]] = {}
    for c in cells:
        if c.get("label_frac", 1.0) != 1.0:
            continue
        by_method.setdefault(c["method"], []).append(c["test"]["aggregate"])
    rows = []
    for method, aggs in by_method.items():
        row: Dict[str, object] = {"method": method,
                                  "label": METHOD_LABELS.get(method, method),
                                  "n_runs": len(aggs)}
        for key in ("dice", "iou", "bf2", "bf5", "hd95", "betti0_err", "mean_steps"):
            vals = [a.get(key) for a in aggs if a.get(key) is not None
                    and np.isfinite(a.get(key, np.nan))]
            row[key] = float(np.mean(vals)) if vals else float("nan")
            row[f"{key}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    order = ["singleshot", "textual_bottleneck", "ptea_lite", "dense_unrolled", "sparcseg"]
    rows.sort(key=lambda r: order.index(r["method"]) if r["method"] in order else 99)
    return rows


def markdown_table(rows: Sequence[Dict], columns: Sequence[Tuple[str, str]],
                   float_fmt: str = "{:.4f}") -> str:
    head = "| " + " | ".join(label for _, label in columns) + " |"
    sep = "|" + "|".join("---" for _ in columns) + "|"
    lines = [head, sep]
    for r in rows:
        cells = []
        for key, _ in columns:
            v = r.get(key)
            if isinstance(v, float):
                cells.append("--" if not np.isfinite(v) else float_fmt.format(v))
            else:
                cells.append(str(v) if v is not None else "--")
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def print_report(results: Dict[str, object]) -> None:
    banner(f"REPORT -- {results['dataset_info']['name']}")

    print("\nT1. Main results\n")
    print(markdown_table(results.get("main_table", []), [
        ("label", "Method"), ("dice", "Dice"), ("iou", "IoU"),
        ("bf2", "BF@2"), ("bf5", "BF@5"), ("hd95", "HD95"),
        ("betti0_err", "beta0 err"), ("mean_steps", "steps"),
    ]))

    sig = results.get("significance")
    if sig:
        print("\n   Paired Wilcoxon vs SPARC-Seg (Holm-corrected):")
        for method, s in sig.items():
            print(f"     {method:>20}  dDice {s['diff']:+.4f} "
                  f"[{s['lo']:+.4f}, {s['hi']:+.4f}]  "
                  f"p_holm {s.get('p_holm', float('nan')):.2e} "
                  f"{stars(s.get('p_holm', float('nan')))}")

    faith = results.get("faithfulness", {})
    if faith:
        key = sorted(faith)[0]
        block = faith[key]
        print("\nT2. Causal faithfulness (norm-matched)\n")
        rows = []
        for method in ("sparcseg", "dense_unrolled"):
            b = block.get(method)
            if not b:
                continue
            c = b["curves"]
            rows.append({
                "label": METHOD_LABELS.get(method, method),
                "CSI": c.get("CSI_normalized"),
                "CSI_tb": c.get("CSI_topbottom"),
                "nec_auc": c.get("necessity_auc"),
                "null_auc": c.get("random_null_auc"),
                "suff_auc": c.get("sufficiency_auc"),
                "naive": c.get("naive_top1_necessity"),
                "units": c.get("n_active_units"),
                "TTI": (b.get("transplant") or {}).get("TTI"),
                "align": (b.get("alignment") or {}).get("alignment_gain"),
                "cev": (b.get("bottleneck") or {}).get("code_explained_variance"),
            })
        print(markdown_table(rows, [
            ("label", "Method"), ("CSI", "CSI (norm.)"), ("CSI_tb", "CSI top-bot"),
            ("nec_auc", "Necessity AUC"), ("null_auc", "Random null AUC"),
            ("suff_auc", "Sufficiency AUC"), ("naive", "naive top-1*"),
            ("units", "active units"), ("cev", "code expl. var"),
            ("TTI", "Transplant TTI"), ("align", "Align. gain"),
        ]))
        print("   * naive top-1 necessity is confounded by ablated-norm; shown "
              "only for comparability with prior work.")
        cmp = block.get("csi_comparison")
        if cmp:
            print(f"\n   CSI gap (sparse - dense): {cmp['gap']:+.4f}  "
                  f"p {cmp['p']:.2e} {stars(cmp['p'])}  effect {cmp['effect']:+.3f}")
        for method in ("sparcseg", "dense_unrolled"):
            b = block.get(method)
            if b and "energy_coupling" in b:
                ec = b["energy_coupling"]
                print(f"   [{method}] monotone-descent rate "
                      f"{ec.get('monotone_descent_rate', float('nan')):.3f}; "
                      f"energy-gain alignment rho "
                      f"{ec.get('energy_gain_alignment', float('nan')):+.3f} "
                      f"(positive = energy drops track Dice gains)")

    eff = results.get("efficiency", {})
    if eff:
        key = sorted(eff)[0]
        print("\nT3. Efficiency (adaptive depth)\n")
        print(markdown_table(eff[key].get("rows", []), [
            ("mode", "Mode"), ("K", "K max"), ("mean_steps", "mean steps"),
            ("dice", "Dice"), ("bf2", "BF@2"), ("ms_per_image", "ms/img"),
        ]))

    con = results.get("concepts", {})
    folds = [k for k in con if k.startswith("fold")]
    if folds:
        c = con[folds[0]]
        print("\nT4. Concept vocabulary\n")
        s = c["summary"]
        print(f"   atoms {int(s['n_atoms'])}, named {int(s['n_named'])} "
              f"({s['named_fraction']:.1%}), dead {int(s['n_dead'])}, "
              f"mean |rho| {s['mean_abs_rho_named']:.3f}")
        print(f"   vocabulary: {s['vocabulary']}")
        if "stability" in con:
            print(f"   naming stability across folds: {con['stability']:.1%}")

    ab = results.get("ablations")
    if ab:
        print("\nT5. Ablations\n")
        print(markdown_table(ab["rows"], [
            ("ablation", "Variant"), ("dice", "Dice"), ("bf2", "BF@2"),
            ("hd95", "HD95"), ("mean_steps", "steps"),
        ]))

In [ ]:
# ===== sparcseg/viz.py =======================================================
"""Figures for the paper.

Five figures carry the argument, and each is generated directly from the same
result objects the tables come from, so a figure can never disagree with a
number in the text:

  F1  qualitative revision -- image, GT, and the mask at every reasoning step
  F2  energy trace + per-step Dice, jointly (the descent-vs-accuracy claim)
  F3  necessity curves: importance-ordered vs the support-restricted null,
      SPARC-Seg against the dense control (the paper's headline figure)
  F4  atom cards -- activation map, name, measured correlation, ablation effect
  F5  accuracy/compute trade-off with adaptive depth marked

Matplotlib only, no seaborn, default colour cycle, one chart per figure.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pass  # (inlined below)
pass  # (inlined below)


def _save(fig, out: Optional[str]):
    if out:
        Path(out).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=180, bbox_inches="tight")
        print(f"saved figure -> {out}")
    return fig


@torch.no_grad()
def figure_revision(model, loader, device, n_images: int = 4,
                    out: Optional[str] = None, threshold: float = 0.5):
    """F1: does the sketch visibly get revised, and does it get better?"""
    model.eval()
    batch = next(iter(loader))
    x = batch["image"][:n_images].to(device)
    gt = batch["mask"][:n_images, 0].numpy() > 0.5
    out_r = model(x, n_steps=model.n_steps, adaptive=False)
    steps = out_r.logits_per_step
    K = len(steps)

    fig, axes = plt.subplots(n_images, K + 2, figsize=(2.0 * (K + 2), 2.0 * n_images))
    axes = np.atleast_2d(axes)
    for i in range(min(n_images, x.shape[0])):
        axes[i, 0].imshow(denormalize(x[i]))
        axes[i, 0].set_ylabel(batch["sample_id"][i][:14], fontsize=7)
        if i == 0:
            axes[i, 0].set_title("image", fontsize=9)
        axes[i, 1].imshow(gt[i], cmap="gray")
        if i == 0:
            axes[i, 1].set_title("ground truth", fontsize=9)
        for t in range(K):
            p = (torch.sigmoid(steps[t][i, 0]) > threshold).cpu().numpy()
            axes[i, t + 2].imshow(p, cmap="gray")
            if i == 0:
                axes[i, t + 2].set_title(f"$S_{{{t + 1}}}$", fontsize=9)
        for a in axes[i]:
            a.set_xticks([]); a.set_yticks([])
    fig.suptitle("Working sketch revised across reasoning steps", fontsize=11)
    fig.tight_layout()
    return _save(fig, out)


@torch.no_grad()
def figure_energy_trace(model, loader, device, n_images: int = 16,
                        out: Optional[str] = None, threshold: float = 0.5):
    """F2: energy falls monotonically; Dice rises alongside it."""
    pass  # (inlined below)
    model.eval()
    batch = next(iter(loader))
    x = batch["image"][:n_images].to(device)
    gt = batch["mask"][:n_images, 0].numpy() > 0.5
    r = model(x, n_steps=model.n_steps, adaptive=False, backtracking=True)
    E = r.energy.stack().numpy()                       # (K+1, B)
    dices = np.stack([[dice_score((torch.sigmoid(lg[i, 0]) > threshold).cpu().numpy(), gt[i])
                       for i in range(x.shape[0])] for lg in r.logits_per_step])

    fig, ax1 = plt.subplots(figsize=(6.0, 3.6))
    steps = np.arange(E.shape[0])
    En = E / np.maximum(E[0:1], 1e-8)
    ax1.plot(steps, En.mean(1), marker="o", color="C0", label="energy $E(z_t,S_t)$")
    ax1.fill_between(steps, np.percentile(En, 25, axis=1),
                     np.percentile(En, 75, axis=1), alpha=0.18, color="C0")
    ax1.set_xlabel("reasoning step $t$")
    ax1.set_ylabel("energy (normalised to $E_0$)", color="C0")
    ax1.tick_params(axis="y", labelcolor="C0")

    ax2 = ax1.twinx()
    ax2.plot(np.arange(1, dices.shape[0] + 1), dices.mean(1), marker="s",
             color="C1", label="Dice")
    ax2.set_ylabel("Dice", color="C1")
    ax2.tick_params(axis="y", labelcolor="C1")
    ax1.set_title("Energy descent and segmentation accuracy, per step")
    ax1.grid(alpha=0.25)
    fig.tight_layout()
    return _save(fig, out)


def figure_necessity_curves(faithfulness_block: Dict, out: Optional[str] = None):
    """F3: the headline figure -- ordered vs null, sparse vs dense."""
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    styles = {"sparcseg": ("C0", "SPARC-Seg"),
              "dense_unrolled": ("C3", "Dense control ($\\lambda_1=0$)")}
    for method, (color, label) in styles.items():
        b = faithfulness_block.get(method)
        if not b:
            continue
        c = b["curves"]
        fr = b.get("fractions", [])
        top = [c.get(f"nec_top@{f}", np.nan) for f in fr]
        rnd = [c.get(f"nec_rand@{f}", np.nan) for f in fr]
        ax.plot(fr, top, marker="o", color=color, label=f"{label}: importance-ordered")
        ax.plot(fr, rnd, marker="x", ls="--", color=color, alpha=0.65,
                label=f"{label}: matched random null")
        ax.fill_between(fr, rnd, top, color=color, alpha=0.12)
    ax.set_xlabel("fraction of reconstruction energy ablated  $\\rho$")
    ax.set_ylabel("$\\Delta$ Dice (drop from unablated)")
    ax.set_title("Norm-matched causal necessity\n(shaded area = CSI)")
    ax.axhline(0.0, color="k", lw=0.7, alpha=0.5)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7.5, loc="best")
    fig.tight_layout()
    return _save(fig, out)


@torch.no_grad()
def figure_atom_cards(model, loader, device, concept_result: Dict,
                      n_atoms: int = 6, n_images: int = 3,
                      out: Optional[str] = None, threshold: float = 0.5):
    """F4: what an atom looks like, what it is named, and what removing it does."""
    model.eval()
    batch = next(iter(loader))
    x = batch["image"][:n_images].to(device)
    r = model(x, n_steps=model.n_steps, adaptive=False)
    z = r.codes[-1]
    base_pred = binarize(r.logits, threshold)

    energies = unit_energy(z)
    top = torch.topk(energies.sum(0), k=min(n_atoms, energies.shape[1])).indices.tolist()
    names = {p["atom"]: p["name"] for p in concept_result.get("profiles", [])}
    rhos = {p["atom"]: p["rho"] for p in concept_result.get("profiles", [])}

    fig, axes = plt.subplots(n_images, len(top) + 1,
                             figsize=(2.0 * (len(top) + 1), 2.1 * n_images))
    axes = np.atleast_2d(axes)
    for i in range(min(n_images, x.shape[0])):
        axes[i, 0].imshow(denormalize(x[i]))
        axes[i, 0].contour(base_pred[i], levels=[0.5], colors="lime", linewidths=1.0)
        if i == 0:
            axes[i, 0].set_title("image + prediction", fontsize=8)
        axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])

    for c, j in enumerate(top):
        keep = torch.ones_like(energies)
        keep[:, j] = 0.0
        abl = model(x, n_steps=model.n_steps,
                    intervene=make_ablation_hook(keep[:, :, None, None], None, model.n_steps))
        abl_pred = binarize(abl.logits, threshold)
        for i in range(min(n_images, x.shape[0])):
            act = z[i, j].cpu().numpy()
            act_up = F.interpolate(torch.tensor(act)[None, None],
                                   size=base_pred.shape[-2:], mode="bilinear",
                                   align_corners=False)[0, 0].numpy()
            ax = axes[i, c + 1]
            ax.imshow(act_up, cmap="magma")
            ax.contour(np.logical_xor(abl_pred[i], base_pred[i]), levels=[0.5],
                       colors="cyan", linewidths=0.9)
            if i == 0:
                nm = names.get(j, "?")
                rho = rhos.get(j, float("nan"))
                ax.set_title(f"#{j} {nm}\n$\\rho$={rho:.2f}", fontsize=7)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Atom activation (magma) and the mask region its removal changes (cyan)",
                 fontsize=10)
    fig.tight_layout()
    return _save(fig, out)


def figure_efficiency(efficiency_block: Dict, out: Optional[str] = None):
    """F5: accuracy per unit of compute, with the adaptive point marked."""
    rows = efficiency_block.get("rows", [])
    if not rows:
        return None
    fixed = [r for r in rows if r["mode"] == "fixed"]
    extra = [r for r in rows if r["mode"].startswith("fixed (extrap")]
    ad = [r for r in rows if r["mode"] == "adaptive"]

    fig, ax = plt.subplots(figsize=(5.8, 3.8))
    if fixed:
        ax.plot([r["ms_per_image"] for r in fixed], [r["dice"] for r in fixed],
                marker="o", color="C0", label="fixed depth (trained range)")
    if extra:
        ax.plot([r["ms_per_image"] for r in extra], [r["dice"] for r in extra],
                marker="o", ls=":", color="C0", alpha=0.5,
                label="fixed depth (extrapolated beyond $K$)")
    if ad:
        ax.scatter([ad[0]["ms_per_image"]], [ad[0]["dice"]], marker="*", s=220,
                   color="C1", zorder=5,
                   label=f"adaptive (mean {ad[0]['mean_steps']:.2f} steps)")
    ax.set_xlabel("inference time (ms / image)")
    ax.set_ylabel("Dice")
    ax.set_title("Accuracy vs. test-time compute")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, out)


def make_all_figures(results: Dict, model=None, loader=None, device=None,
                     out_dir: str = "/kaggle/working/sparcseg_figures") -> List[str]:
    """Generate whatever the available results support; skip the rest quietly."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    made: List[str] = []

    faith = results.get("faithfulness", {})
    if faith:
        key = sorted(faith)[0]
        p = f"{out_dir}/F3_necessity_curves.png"
        if figure_necessity_curves(faith[key], p):
            made.append(p)

    eff = results.get("efficiency", {})
    if eff:
        key = sorted(eff)[0]
        p = f"{out_dir}/F5_efficiency.png"
        if figure_efficiency(eff[key], p):
            made.append(p)

    if model is not None and loader is not None and device is not None:
        p = f"{out_dir}/F1_revision.png"
        figure_revision(model, loader, device, out=p); made.append(p)
        p = f"{out_dir}/F2_energy_trace.png"
        figure_energy_trace(model, loader, device, out=p); made.append(p)
        con = results.get("concepts", {})
        folds = [k for k in con if k.startswith("fold")]
        if folds:
            prof = {"profiles": con[folds[0]].get("top", [])}
            p = f"{out_dir}/F4_atom_cards.png"
            figure_atom_cards(model, loader, device, prof, out=p); made.append(p)
    return made

---
## Part 5 - Run it

### 5.1 Locate the data

Discovery is by file-signature matching rather than a hard-coded Kaggle slug, because the same dataset is mirrored under many slugs and nested at unpredictable depths. If it fails, the error lists what *is* mounted.

In [ ]:
DATASET = 'brisc'
ROOT = None   # set explicitly only if auto-discovery picks the wrong folder

samples = load_samples(DATASET, ROOT)
print(f'found {len(samples)} image/mask pairs')
print(summarize_samples(samples))
print('\nfirst three:')
for s in samples[:3]:
    print(f'  {s.sample_id:<28} {s.image_path.name:<28} '
          f'masks={[m.name for m in s.mask_paths]}')

### 5.2 Look at the data before trusting any number

Empty-mask rate is the single most important thing to check: it sets the floor Dice a model gets for predicting nothing, and it differs between these three datasets.

In [ ]:
import matplotlib.pyplot as plt

ds_peek = SegmentationDataset(samples, img_size=256, train=False)
idx = np.linspace(0, len(samples) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
empty = 0
for c, i in enumerate(idx):
    img, msk = ds_peek._load_resized(int(i))
    axes[0, c].imshow(img); axes[0, c].set_title(samples[i].sample_id[:18], fontsize=8)
    axes[1, c].imshow(msk, cmap='gray')
    for a in (axes[0, c], axes[1, c]):
        a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

areas = []
for i in range(0, len(samples), max(1, len(samples) // 300)):
    _, m = ds_peek._load_resized(i)
    areas.append(float((m > 0).mean()))
areas = np.array(areas)
print(f'lesion area: median {np.median(areas):.3%}, '
      f'p10 {np.percentile(areas, 10):.3%}, p90 {np.percentile(areas, 90):.3%}')
print(f'empty-mask rate: {(areas == 0).mean():.1%}  '
      f'<- a model predicting nothing scores this as Dice')

### 5.3 Smoke test (~2 minutes)

Runs the whole pipeline at toy scale. **Run this before the real thing.** It catches a wrong data path, a broken GPU, or an environment problem in two minutes instead of two hours, and it verifies the descent property the paper's Proposition claims.

In [ ]:
smoke_cfg = CoreConfig(img_size=128, sketch_dim=32, dict_size=64, n_steps=3,
                       topk_atoms=6, backbone='resnet18', pretrained=False,
                       batch_size=8, epochs=2, num_workers=2,
                       causal_max_images=16, n_random_controls=2,
                       n_transplant_pairs=16, bootstrap_n=200,
                       ablation_fractions=(0.1, 0.4))
smoke_plan = ExperimentPlan(methods=('singleshot', 'dense_unrolled', 'sparcseg'),
                            folds=(0,), seeds=(0,), epochs=2,
                            run_steering=False, run_concepts=False,
                            run_ablations=False, max_train_images=64)

_smoke = run_dataset_experiment(DATASET, smoke_cfg, smoke_plan, root=ROOT,
                                out_dir='/kaggle/working/smoke')
print_report(_smoke)

_f = _smoke.get('faithfulness', {})
if _f:
    _b = _f[sorted(_f)[0]].get('sparcseg', {})
    _rate = _b.get('energy_coupling', {}).get('monotone_descent_rate')
    print(f'\nmonotone-descent rate: {_rate}  (must be 1.0 - this is the '
          f'Proposition, audited numerically)')
    assert _rate is None or _rate >= 0.999, 'ENERGY INCREASED - do not trust any result below'

### 5.4 The real run

`quick_plan` is one fold, all five methods, the full causal protocol - enough for a complete (if single-fold) results section. `full_plan` is the paper configuration: 5 folds x 3 seeds plus every ablation, which will not fit in one Kaggle session - split it by passing `folds=` and `methods=` and merge the saved JSONs afterwards.

**The `(m, K)` decision is already locked**: `CoreConfig` is fixed across all three datasets, giving the stronger 'one architecture, three domains' claim. Per-dataset tuning stays available as a supplementary ablation.

In [ ]:
cfg = CoreConfig()          # shared verbatim across BUSI / ISIC / BRISC
plan = quick_plan(epochs=20)

# For the paper configuration instead:
# plan = full_plan()
# For one Kaggle session at a time:
# plan = ExperimentPlan(folds=(0, 1), seeds=(0,), run_ablations=True,
#                       low_label_fracs=(0.1, 0.25, 0.5),
#                       dict_sizes=(64, 128, 256), step_counts=(1, 2, 6),
#                       topk_values=(2, 4, 16, 32))

results = run_dataset_experiment(DATASET, cfg, plan, root=ROOT,
                                 out_dir='/kaggle/working/sparcseg_results')

### 5.5 Tables

In [ ]:
print_report(results)

### 5.6 Figures

Generated from the same result objects the tables come from, so a figure can never disagree with a number in the text.

In [ ]:
cells_ = results.get('_last_cells', {})
sp = cells_.get('sparcseg')
device = get_device()
model = loader = None
if sp is not None:
    model = sp['model'].to(device)
    loader = sp['loaders'][2]

paths = make_all_figures(results, model, loader, device,
                         out_dir='/kaggle/working/sparcseg_figures')

from IPython.display import Image, display
for p in paths:
    display(Image(filename=p))

### 5.7 What to check before believing the result

In order of how badly each one invalidates the paper:

1. **`monotone_descent_rate == 1.000`.** If not, the Proposition is false on this run and the energy story goes with it.
2. **`code expl. var` is substantially above 0** for SPARC-Seg. If the code explains almost none of the sketch, the readout is decoding something the code did not build, no intervention can matter, and a null CSI is uninterpretable. Lower `lambda_evidence` if this is small.
3. **`active units` is close to `topk_atoms`.** If it drifts up toward `dict_size`, sparsity has collapsed and the audit story with it.
4. **CSI(SPARC-Seg) > CSI(dense), significantly.** This is the claim. Note that the *naive* top-1 column can favour either model - that is the confound, demonstrated rather than argued.
5. **`n_dead` atoms.** A large dead fraction means `dict_size` is larger than the data needs; the `dict_sizes` ablation is the honest answer.
6. **`collapsed_to_empty` warnings.** Usually too few epochs, or a low-label setting that needs more.

A null CSI result is publishable at this workshop *if* checks 1-3 pass: it would say that constructive sparsity does not by itself buy causal faithfulness in dense prediction, which is a real finding about a field that currently assumes otherwise. Reporting it as a null needs those three checks to rule out the boring explanations.

In [ ]:
import json, os
print('written to /kaggle/working:')
for dirpath, _, files in os.walk('/kaggle/working'):
    for f in sorted(files):
        p = os.path.join(dirpath, f)
        print(f'  {p}  ({os.path.getsize(p) / 1024:.0f} KB)')

# results.json holds every per-image score, so the paired tests and the
# cross-dataset aggregation can be redone later without retraining.
print('\nCommit this notebook so /kaggle/working persists as a version output,\n'
      'then hand results.json to whoever is aggregating the three datasets.')